# AUR–QAUR Portfolio Research — Data NCKH Audit 5/9

Notebook Colab hoàn chỉnh để tái lập hệ thống **AI–Quantum Portfolio Optimization** từ bộ dữ liệu đã audit ngày 5/9. Notebook tải đúng `data_5_9.zip` từ GitHub, kiểm tra SHA-256 trước khi chạy, dựng toàn bộ mã nguồn trong runtime và sinh lại bảng kết quả, kiểm định, biểu đồ cùng danh mục cuối.

**Cách chạy:** chọn `Runtime → Run all` trên Google Colab CPU. Không cần clone repository, mount Google Drive hoặc sửa đường dẫn. Thời gian tham khảo khoảng 15–25 phút tùy runtime.

> Phạm vi học thuật: dữ liệu đến 2025-12-31 là panel lịch sử đã audit; phần 2026 đến 2026-08-28 là mở rộng quan sát tạm thời. Notebook không tuyên bố lợi thế lượng tử và không cho phép triển khai vốn thật.


## Mục lục và luồng thực thi

| Phần | Nội dung | Đầu ra chính |
|---:|---|---|
| 0 | Cài đặt, cấu hình và seed | Môi trường tái lập |
| 1 | Tải và kiểm định Data NCKH Audit 5/9 | Hash, schema, thống kê dữ liệu |
| 2–3 | Khởi tạo mã nguồn và bộ điều phối | Hai module Python độc lập |
| 4 | Chạy toàn bộ pipeline | Thư mục kết quả hiện tại |
| 5–10 | Sáu giai đoạn phương pháp | Forecast, AUR, QAUR, QUBO, backtest |
| 11–14 | Thiết kế thực tiễn và kết luận | Robustness, basket, figures |
| 15 | Audit fail-fast | 21 kiểm tra bắt buộc |
| 16 | Đóng gói kết quả | ZIP tải về tùy chọn |

Mỗi phần gồm một cell Markdown giải thích mục tiêu và một hoặc nhiều cell code thực hiện đúng nhiệm vụ đó. Nếu một kiểm tra quan trọng thất bại, notebook dừng ngay thay vì tiếp tục với dữ liệu sai.


## Kiến trúc phương pháp luận và ký hiệu

\[
\mathcal U_t
\xrightarrow{\text{XGBoost/EWMA}}
(\hat\mu_t,\hat\Sigma_t)
\begin{cases}
\xrightarrow{\mathrm{AUR}}\mathrm{Top}\text{-}K_A\\
\xrightarrow{\mathrm{QAUR}\;(Q^{UR})}\mathrm{Top}\text{-}K_{QA}
\end{cases}
\xrightarrow{\text{cùng }Q^{PO}\text{ và XY-QAOA}}
k_p\text{ tài sản}
\xrightarrow{\text{classical weights}}
w_t
\xrightarrow{\text{walk-forward}}
r^{OOS}_{t+1}.
\]

Hai reducer nhận cùng point-in-time universe \(\mathcal U_t\), cùng forecast
\(\hat\mu_t\), cùng EWMA covariance \(\hat\Sigma_t\), cùng lịch folds và cùng
các giả định downstream. Vì vậy, chênh lệch giữa hai nhánh được quy chủ yếu cho
cơ chế universe reduction. Framework gồm đúng sáu giai đoạn:

1. Dự báo lợi suất và rủi ro;
2. Adaptive Universe Reduction;
3. Quantum-Assisted Universe Reduction;
4. So sánh hai phương pháp giảm vũ trụ;
5. Portfolio Optimization dùng chung;
6. Walk-Forward Backtest ngoài mẫu.


## 0. Cài đặt và cấu hình môi trường

Cell đầu cài các thư viện khoa học dùng trong Colab. Cell cấu hình kế tiếp tập trung URL, checksum, seed và đường dẫn runtime tại một nơi để dễ audit; không có tham số ẩn trong các cell phía sau.


In [ ]:
# Cài đặt phiên bản tối thiểu tương thích với Google Colab CPU.
%pip -q install "numpy>=2.0" "pandas>=2.2" "scipy>=1.13" "scikit-learn>=1.5" "xgboost>=2.1" "matplotlib>=3.8" "pyarrow>=16" "tabulate>=0.9"


In [ ]:
from dataclasses import dataclass
from pathlib import Path
import os
import random

import numpy as np

@dataclass(frozen=True)
class NotebookConfig:
    # Bản ZIP đã commit trên GitHub. Tên file dùng 5_9 vì Windows không cho phép dấu '/' trong tên.
    data_url: str = "https://raw.githubusercontent.com/23022006muki/AI-Quantum---Finance-Portfolio-Optimization/main/Data%205_9/data_5_9.zip"
    zip_sha256: str = "e1a9d2c1997fb9ad6b3127eb4a1e175a2f74fa3678ed568a91a47d5aae44fad9"
    csv_sha256: str = "b0a16d9f8c31a2a5d4e1ba8f00d49b50f112f149d4fae23b3529df085a45ccb2"
    csv_filename: str = "data_5_9.csv"
    workdir: Path = Path("/content/aur_qaur_data_nckh_5_9")
    random_seed: int = 42
    transaction_cost_bps: float = 25.0

CONFIG = NotebookConfig()
WORKDIR = CONFIG.workdir
WORKDIR.mkdir(parents=True, exist_ok=True)

# Seed thống nhất cho các thành phần ngẫu nhiên dùng trong Python và NumPy.
os.environ["PYTHONHASHSEED"] = str(CONFIG.random_seed)
random.seed(CONFIG.random_seed)
np.random.seed(CONFIG.random_seed)

print(CONFIG)
print("Runtime directory:", WORKDIR)


## 1. Nạp dữ liệu và kiểm tra chất lượng đầu vào

Notebook tải bản ZIP công khai từ GitHub, kiểm tra hash của ZIP **trước khi giải nén**, sau đó kiểm tra hash CSV. Nếu mạng GitHub tạm thời không khả dụng, cell cho phép tải thủ công đúng file `data_5_9.zip`. Không có dữ liệu synthetic trong thực nghiệm chính.


In [ ]:
from pathlib import Path
import hashlib
import urllib.request
import zipfile

DATA_ZIP = WORKDIR / "data_5_9.zip"

def sha256_file(path: Path) -> str:
    """Tính SHA-256 theo block để không nạp toàn bộ file lớn vào RAM."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

try:
    print("Đang tải dữ liệu từ GitHub...")
    urllib.request.urlretrieve(CONFIG.data_url, DATA_ZIP)
except Exception as download_error:
    print("Tải tự động thất bại:", download_error)
    print("Hãy chọn đúng file data_5_9.zip khi hộp thoại tải lên xuất hiện.")
    from google.colab import files
    uploaded = files.upload()
    if "data_5_9.zip" not in uploaded:
        raise RuntimeError("Không tìm thấy data_5_9.zip trong tệp đã tải lên.")
    DATA_ZIP.write_bytes(uploaded["data_5_9.zip"])

zip_digest = sha256_file(DATA_ZIP)
assert zip_digest == CONFIG.zip_sha256, f"ZIP SHA-256 không khớp: {zip_digest}"

with zipfile.ZipFile(DATA_ZIP) as archive:
    assert archive.namelist() == [CONFIG.csv_filename], archive.namelist()
    assert archive.testzip() is None, "ZIP không vượt qua kiểm tra CRC."
    archive.extractall(WORKDIR)

DATASET = WORKDIR / CONFIG.csv_filename
digest = sha256_file(DATASET)
assert digest == CONFIG.csv_sha256, f"CSV SHA-256 không khớp: {digest}"

print("Dataset:", DATASET)
print("Dung lượng:", f"{DATASET.stat().st_size / 1024**2:.2f} MiB")
print("ZIP SHA-256:", zip_digest)
print("CSV SHA-256:", digest)


In [ ]:
import platform
import numpy as np
import pandas as pd
import scipy
import sklearn
import xgboost
from IPython.display import display, Markdown

# Đọc một lần toàn bộ bảng chuẩn; các record_type có schema riêng trong cùng CSV.
raw_data = pd.read_csv(DATASET, low_memory=False)
price_raw = raw_data.loc[raw_data["record_type"].eq("PRICE")].copy()
security_raw = raw_data.loc[raw_data["record_type"].eq("SECURITY")].copy()
benchmark_raw = raw_data.loc[raw_data["record_type"].eq("BENCHMARK")].copy()
action_raw = raw_data.loc[raw_data["record_type"].eq("CORPORATE_ACTION")].copy()
price_raw["date"] = pd.to_datetime(price_raw["date"], errors="coerce")

# Kiểm tra các điều kiện đầu vào thực sự cần cho mô hình giá.
duplicate_price_keys = int(price_raw.duplicated(["ticker", "date"]).sum())
missing_price_contract = price_raw[["date", "ticker", "adjusted_close", "volume", "trading_value"]].isna().sum()
invalid_adjusted_close = int((pd.to_numeric(price_raw["adjusted_close"], errors="coerce") <= 0).sum())
invalid_volume = int((pd.to_numeric(price_raw["volume"], errors="coerce") < 0).sum())
invalid_trading_value = int((pd.to_numeric(price_raw["trading_value"], errors="coerce") < 0).sum())

record_counts = raw_data["record_type"].value_counts().sort_index().rename_axis("record_type").reset_index(name="rows")
data_overview = pd.DataFrame([
    {"indicator": "Total records", "value": len(raw_data)},
    {"indicator": "Price records", "value": len(price_raw)},
    {"indicator": "Security records", "value": len(security_raw)},
    {"indicator": "Corporate-action records", "value": len(action_raw)},
    {"indicator": "Benchmark records", "value": len(benchmark_raw)},
    {"indicator": "Unique price tickers", "value": price_raw["ticker"].nunique()},
    {"indicator": "First price date", "value": str(price_raw["date"].min().date())},
    {"indicator": "Last observed price date", "value": str(price_raw["date"].max().date())},
    {"indicator": "Duplicate price keys", "value": duplicate_price_keys},
    {"indicator": "Invalid adjusted close", "value": invalid_adjusted_close},
    {"indicator": "Invalid volume", "value": invalid_volume},
    {"indicator": "Invalid trading value", "value": invalid_trading_value},
    {"indicator": "Dataset SHA-256", "value": digest},
])
environment = pd.DataFrame([
    {"component": "Python", "version": platform.python_version()},
    {"component": "NumPy", "version": np.__version__},
    {"component": "pandas", "version": pd.__version__},
    {"component": "SciPy", "version": scipy.__version__},
    {"component": "scikit-learn", "version": sklearn.__version__},
    {"component": "XGBoost", "version": xgboost.__version__},
])

# Fail-fast: mọi sai lệch quan trọng đều dừng notebook trước bước mô hình.
assert len(raw_data) == 179_173
assert len(price_raw) == 174_626
assert price_raw["ticker"].nunique() == 120
assert duplicate_price_keys == 0
assert int(missing_price_contract.sum()) == 0
assert invalid_adjusted_close == invalid_volume == invalid_trading_value == 0

display(Markdown("### Quy mô và phạm vi dữ liệu"))
display(data_overview)
display(Markdown("### Phân bố theo loại bản ghi"))
display(record_counts)
display(Markdown("### Thiếu dữ liệu trong hợp đồng PRICE"))
display(missing_price_contract.rename("missing_values").to_frame())
display(Markdown("### Môi trường thực thi"))
display(environment)
print("DATA_INPUT_AUDIT_OK")


## 2. Toàn bộ source code của phương pháp

Cell dưới đây viết trực tiếp experimental engine vào runtime. Source bao gồm
feature engineering, purged walk-forward forecast, XGBoost, EWMA, point-in-time
eligibility, AUR, QAUR, \(Q^{UR}\), \(Q^{PO}\), exact fixed-cardinality
reference, XY-QAOA statevector audit, bounded weight allocation, turnover,
transaction costs, market gate và các kiểm định thống kê.

### Giai đoạn 1 — Dự báo lợi suất và rủi ro

\[
y_{i,t}^{(h)}=\frac{P_{i,t+h}}{P_{i,t}}-1,\qquad
\hat\Sigma_t=(1-\lambda)\sum_{\tau\le t}\lambda^{t-\tau}
(r_\tau-\bar r_t)(r_\tau-\bar r_t)^\top.
\]

Training labels được purge trước decision time. Validation Rank IC chỉ là chẩn
đoán năng lực xếp hạng và không được dùng như một quan sát độc lập về alpha.

### Giai đoạn 2 — Adaptive Universe Reduction

\[
s_{i,t}=w_sS_{i,t}+w_lL_{i,t}+w_rR_{i,t}+w_hH_{i,t}.
\]

AUR xây candidate set tuần tự bằng cách cân bằng unary quality với incremental
correlation redundancy.

### Giai đoạn 3 — Quantum-Assisted Universe Reduction

\[
\max_{z}\;Q^{UR}(z)=\sum_i s_{i,t}z_i-lambda_c
\sum_{i<j}|\rho_{ij,t}|z_iz_j,qquad \sum_i z_i=K.
\]

QAUR đánh giá đồng thời các quan hệ cặp trong feasible cardinality subspace.
Backend hiện tại là classical cardinality-preserving surrogate cho
quantum-ready QUBO; notebook không tuyên bố quantum advantage.

### Giai đoạn 4 — So sánh reducer

\[
J_t=\frac{|\mathrm{Top}\text{-}K_A\cap\mathrm{Top}\text{-}K_{QA}|}
{|\mathrm{Top}\text{-}K_A\cup\mathrm{Top}\text{-}K_{QA}|}.
\]

Ngoài Jaccard, so sánh còn sử dụng \(Q^{UR}\), mean absolute correlation và
candidate turnover.

### Giai đoạn 5 — Portfolio Optimization dùng chung

\[
\min_x\;x^\top Q^{PO}x,\qquad
Q^{PO}=\lambda_p\widetilde\Sigma-\operatorname{diag}(\widetilde\mu),
\qquad \sum_i x_i=k_p.
\]

XY-QAOA dùng fixed-Hamming-weight initialization và constraint-preserving XY
mixer. Cùng depth, budget và shots được áp dụng cho hai reducer.

### Giai đoạn 6 — Walk-Forward Backtest

\[
\mathrm{TO}_t=\frac12\sum_i|w_{i,t}-w_{i,t^-}|,\qquad
r^{net}_{t,1}=r^{gross}_{t,1}-\mathrm{TO}_t\frac{c_{bps}}{10^4}.
\]


In [ ]:
%%writefile /content/aur_qaur_data_nckh_5_9/run_constraint_strategy_search.py
from __future__ import annotations

"""Time-respecting strategy search for the AUR-versus-QAUR framework.

The search deliberately separates early development folds from a final temporal
holdout.  Configurations are selected on the average AUR/QAUR development
score, never on the final holdout and never in favour of only one reducer.

Grid screening uses exact enumeration inside the fixed-cardinality feasible
subspace.  The chosen configuration is then audited with the same ideal
fixed-Hamming-weight XY-QAOA statevector simulator used by the notebook.
"""

import argparse
import hashlib
import json
import math
import platform
import time
from dataclasses import asdict, dataclass
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor


SEED = 42
FEATURE_COLUMNS = [
    "return_5d", "return_20d", "return_60d", "return_120d",
    "sma_ratio_20", "ema_ratio_20", "rsi_14", "macd_scaled",
    "volatility_20d", "downside_volatility_20d", "drawdown_60d",
    "liquidity_20d",
]


@dataclass(frozen=True)
class StrategyConfig:
    config_id: str
    family: str
    candidate_size: int
    portfolio_cardinality: int
    weight_upper: float
    weight_lower: float
    weight_mode: str
    signal_blend: float = 1.0
    correlation_penalty: float = 0.10
    stability_weight: float = 0.15
    covariance_span: int = 60
    covariance_shrinkage: float = 0.0
    risk_aversion_qubo: float = 0.55
    risk_aversion_weights: float = 1.25
    turnover_penalty: float = 0.0
    volatility_target: float = 0.0
    transaction_cost_bps: float = 25.0
    qa_warm_start: bool = False
    market_regime_lookback: int = 0
    minimum_validation_ic: float = -1.0


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def rsi(series: pd.Series, window: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(window).mean()
    loss = -delta.clip(upper=0).rolling(window).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - 100 / (1 + rs)


def build_features(prices: pd.DataFrame, horizon: int = 20) -> pd.DataFrame:
    parts: list[pd.DataFrame] = []
    for _, group in prices.groupby("ticker", sort=False):
        g = group.sort_values("date").copy()
        px = g["adjusted_close"].astype(float)
        ret1 = px.pct_change(fill_method=None)
        g["return_1d"] = ret1
        for window in (5, 20, 60, 120):
            g[f"return_{window}d"] = px.pct_change(window, fill_method=None)
        g["sma_ratio_20"] = px / px.rolling(20).mean() - 1
        g["ema_ratio_20"] = px / px.ewm(span=20, adjust=False).mean() - 1
        g["rsi_14"] = rsi(px, 14) / 100.0
        ema12 = px.ewm(span=12, adjust=False).mean()
        ema26 = px.ewm(span=26, adjust=False).mean()
        g["macd_scaled"] = (ema12 - ema26) / px
        g["volatility_20d"] = ret1.rolling(20).std(ddof=1)
        g["downside_volatility_20d"] = ret1.where(ret1 < 0, 0).rolling(20).std(ddof=1)
        g["drawdown_60d"] = px / px.rolling(60).max() - 1
        liquidity = g["trading_value"].where(g["trading_value"] > 0, g["volume"] * px)
        g["liquidity_20d"] = liquidity.rolling(20).mean()
        g["target_return_20d"] = px.shift(-horizon) / px - 1
        g["target_available_at"] = g["date"].shift(-horizon)
        parts.append(g)
    out = pd.concat(parts, ignore_index=True)
    out["target_rank"] = out.groupby("date")["target_return_20d"].rank(pct=True)
    return out.sort_values(["date", "ticker"]).reset_index(drop=True)


def make_folds(dates: pd.Series) -> list[dict]:
    start = pd.Timestamp(dates.min()).normalize()
    end = pd.Timestamp(dates.max()).normalize()
    train_start = start
    train_end = train_start + pd.DateOffset(months=24)
    folds: list[dict] = []
    fold_id = 0
    while True:
        validation_start = train_end
        validation_end = validation_start + pd.DateOffset(months=3)
        test_start = validation_end
        test_end = test_start + pd.DateOffset(months=1)
        if test_end > end + pd.Timedelta(days=1):
            break
        folds.append({
            "fold": fold_id,
            "train_start": train_start,
            "train_end": train_end,
            "validation_start": validation_start,
            "validation_end": validation_end,
            "test_start": test_start,
            "test_end": test_end,
        })
        fold_id += 1
        train_start += pd.DateOffset(months=1)
        train_end += pd.DateOffset(months=1)
    return folds


def prepare_matrix(frame: pd.DataFrame, medians: pd.Series | None = None):
    x = frame[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
    if medians is None:
        medians = x.median().fillna(0.0)
    return x.fillna(medians).to_numpy(float), medians


def fit_xgboost(train: pd.DataFrame, seed: int):
    usable = train.dropna(subset=["target_rank"])
    x, medians = prepare_matrix(usable)
    model = XGBRegressor(
        n_estimators=120,
        max_depth=3,
        learning_rate=0.035,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        objective="reg:squarederror",
        n_jobs=-1,
        random_state=seed,
    )
    model.fit(x, usable["target_rank"].to_numpy(float))
    return model, medians


def rank01(series: pd.Series, higher_is_better: bool = True) -> pd.Series:
    ranked = series.rank(method="average", pct=True)
    return ranked if higher_is_better else 1.0 - ranked


def financial_metrics(returns: pd.Series) -> dict:
    r = pd.Series(returns).dropna()
    if r.empty:
        return {
            "observations": 0, "cumulative_return": np.nan, "annualized_return": np.nan,
            "annualized_volatility": np.nan, "sharpe_zero_rf": np.nan,
            "sortino_zero_rf": np.nan, "maximum_drawdown": np.nan,
        }
    wealth = (1 + r).cumprod()
    annualized_return = float(wealth.iloc[-1] ** (252 / len(r)) - 1)
    annualized_volatility = float(r.std(ddof=1) * np.sqrt(252))
    downside = float(r[r < 0].std(ddof=1) * np.sqrt(252))
    drawdown = wealth / wealth.cummax() - 1
    return {
        "observations": len(r),
        "cumulative_return": float(wealth.iloc[-1] - 1),
        "annualized_return": annualized_return,
        "annualized_volatility": annualized_volatility,
        "sharpe_zero_rf": annualized_return / annualized_volatility if annualized_volatility > 0 else np.nan,
        "sortino_zero_rf": annualized_return / downside if downside > 0 else np.nan,
        "maximum_drawdown": float(drawdown.min()),
    }


def load_market_data(dataset_path: Path):
    columns = [
        "record_type", "date", "ticker", "adjusted_close", "volume", "trading_value",
        "total_return_index", "listing_date", "delisting_date",
    ]
    raw = pd.read_csv(dataset_path, usecols=columns, low_memory=False)
    prices = raw[raw["record_type"].eq("PRICE")].copy()
    prices["date"] = pd.to_datetime(prices["date"], errors="coerce")
    for column in ("adjusted_close", "volume", "trading_value"):
        prices[column] = pd.to_numeric(prices[column], errors="coerce")
    prices = prices.dropna(subset=["date", "ticker", "adjusted_close"])
    prices = prices[prices["adjusted_close"] > 0].drop_duplicates(["ticker", "date"], keep="last")

    security = raw[raw["record_type"].eq("SECURITY")].copy()
    security["listing_date"] = pd.to_datetime(security["listing_date"], errors="coerce")
    security["delisting_date"] = pd.to_datetime(security["delisting_date"], errors="coerce")
    security = security.dropna(subset=["ticker"]).drop_duplicates("ticker", keep="last").set_index("ticker")

    benchmark = raw[raw["record_type"].eq("BENCHMARK")].copy()
    benchmark["date"] = pd.to_datetime(benchmark["date"], errors="coerce")
    benchmark["total_return_index"] = pd.to_numeric(benchmark["total_return_index"], errors="coerce")
    benchmark = benchmark.dropna(subset=["date", "total_return_index"]).sort_values("date")
    benchmark = benchmark.drop_duplicates("date", keep="last")
    benchmark["return"] = benchmark["total_return_index"].pct_change(fill_method=None)
    return prices, security, benchmark


def point_in_time_eligible(ticker: str, decision_time: pd.Timestamp, security: pd.DataFrame) -> bool:
    if ticker not in security.index:
        return True
    row = security.loc[ticker]
    listing = row["listing_date"]
    delisting = row["delisting_date"]
    return not ((pd.notna(listing) and listing > decision_time) or (pd.notna(delisting) and delisting <= decision_time))


def build_fold_cache(features: pd.DataFrame, security: pd.DataFrame, folds: list[dict], output_dir: Path):
    snapshots: list[pd.DataFrame] = []
    diagnostics: list[dict] = []
    for fold in folds:
        fold_id = fold["fold"]
        started = time.perf_counter()
        train = features[
            (features["date"] >= fold["train_start"])
            & (features["date"] < fold["train_end"])
            & (features["target_available_at"] < fold["train_end"])
        ]
        model, medians = fit_xgboost(train, SEED + fold_id)
        decision_frame = features[features["date"] < fold["test_start"]]
        decision_time = decision_frame["date"].max()
        snapshot = decision_frame[decision_frame["date"].eq(decision_time)].copy()
        snapshot = snapshot[snapshot["ticker"].map(lambda x: point_in_time_eligible(x, decision_time, security))]
        history_start = decision_time - pd.Timedelta(days=252)
        history = features[(features["date"] <= decision_time) & (features["date"] >= history_start)]
        counts = history.groupby("ticker")["return_1d"].count()
        snapshot = snapshot[snapshot["ticker"].isin(counts[counts >= 126].index)].copy()
        x_snapshot, _ = prepare_matrix(snapshot, medians)
        prediction = model.predict(x_snapshot)
        snapshot["xgb_signal"] = pd.Series(prediction, index=snapshot.index).rank(pct=True)
        momentum = 0.5 * rank01(snapshot["return_20d"].fillna(0.0)) + 0.5 * rank01(snapshot["return_60d"].fillna(0.0))
        snapshot["momentum_signal"] = momentum
        snapshot["fold"] = fold_id
        snapshot["decision_time"] = decision_time
        snapshots.append(snapshot[[
            "fold", "decision_time", "ticker", "xgb_signal", "momentum_signal",
            "liquidity_20d", "volatility_20d", "return_20d", "return_60d",
        ]])

        validation = features[
            (features["date"] >= fold["validation_start"])
            & (features["date"] < fold["validation_end"])
            & (features["target_available_at"] < fold["validation_end"])
        ].dropna(subset=["target_rank"])
        x_validation, _ = prepare_matrix(validation, medians)
        val_pred = model.predict(x_validation)
        scored = validation[["date", "target_rank"]].copy()
        scored["prediction"] = val_pred
        daily_ic = scored.groupby("date").apply(
            lambda g: stats.spearmanr(g["prediction"], g["target_rank"], nan_policy="omit").statistic,
            include_groups=False,
        )
        diagnostics.append({
            "fold": fold_id,
            "decision_time": decision_time,
            "validation_rank_ic": float(daily_ic.mean()),
            "validation_rmse": float(mean_squared_error(validation["target_rank"], val_pred) ** 0.5),
            "train_rows": len(train),
            "universe_size": len(snapshot),
            "runtime_seconds": time.perf_counter() - started,
        })
        print(f"forecast fold {fold_id + 1:02d}/{len(folds)} done; universe={len(snapshot)}", flush=True)
    snapshot_table = pd.concat(snapshots, ignore_index=True)
    diagnostic_table = pd.DataFrame(diagnostics)
    snapshot_table.to_csv(output_dir / "forecast_snapshots.csv", index=False)
    diagnostic_table.to_csv(output_dir / "forecast_diagnostics.csv", index=False)
    return snapshot_table, diagnostic_table


def absolute_correlation(return_panel: pd.DataFrame, tickers: list[str], decision_time: pd.Timestamp) -> np.ndarray:
    panel = return_panel.loc[
        (return_panel.index <= decision_time) & (return_panel.index >= decision_time - pd.Timedelta(days=252)),
        tickers,
    ]
    corr = panel.corr(min_periods=20).fillna(0.0).abs().to_numpy(float)
    np.fill_diagonal(corr, 0.0)
    return corr


def common_scores(snapshot: pd.DataFrame, previous: set[str], config: StrategyConfig) -> pd.DataFrame:
    x = snapshot.copy().sort_values("ticker").reset_index(drop=True)
    x["signal"] = config.signal_blend * x["xgb_signal"] + (1 - config.signal_blend) * x["momentum_signal"]
    x["signal_component"] = rank01(x["signal"])
    x["liquidity_component"] = rank01(x["liquidity_20d"].fillna(0.0))
    x["risk_component"] = rank01(x["volatility_20d"].fillna(np.inf), False)
    x["stability_component"] = x["ticker"].isin(previous).astype(float)
    x["unary_score"] = (
        0.40 * x["signal_component"]
        + 0.30 * x["liquidity_component"]
        + 0.15 * x["risk_component"]
        + config.stability_weight * x["stability_component"]
    )
    return x


def reduction_objective(bits: np.ndarray, unary: np.ndarray, corr: np.ndarray, penalty: float) -> float:
    return float(unary @ bits - penalty * 0.5 * bits @ corr @ bits)


def reduce_universe(method: str, snapshot: pd.DataFrame, return_panel: pd.DataFrame, previous: set[str], config: StrategyConfig, seed: int):
    x = common_scores(snapshot, previous, config)
    tickers = x["ticker"].tolist()
    unary = x["unary_score"].to_numpy(float)
    corr = absolute_correlation(return_panel, tickers, pd.Timestamp(snapshot["decision_time"].iloc[0]))
    k = min(config.candidate_size, len(x))
    greedy_selected: list[int] = []
    greedy_remaining = set(range(len(x)))
    while len(greedy_selected) < k:
        best = max(
            greedy_remaining,
            key=lambda i: (
                unary[i] - config.correlation_penalty * corr[i, greedy_selected].sum(),
                tickers[i],
            ),
        )
        greedy_selected.append(best)
        greedy_remaining.remove(best)
    if method == "AUR":
        bits = np.zeros(len(x), dtype=int)
        bits[greedy_selected] = 1
    else:
        rng = np.random.default_rng(seed)
        starts = [np.argsort(unary)[-k:]]
        # A warm-started QAUR is a legitimate hybrid algorithm: it receives the
        # AUR feasible solution and is only allowed to keep or improve it under
        # the exact same Q^UR objective for the current fold.
        if config.qa_warm_start:
            starts.append(np.asarray(greedy_selected, dtype=int))
        starts += [rng.choice(len(x), size=k, replace=False) for _ in range(5)]
        bits, best_value = None, -np.inf
        for start in starts:
            trial = np.zeros(len(x), dtype=int)
            trial[np.asarray(start, dtype=int)] = 1
            value = reduction_objective(trial, unary, corr, config.correlation_penalty)
            for _ in range(40):
                inside = np.flatnonzero(trial)
                outside = np.flatnonzero(1 - trial)
                move, best_delta = None, 0.0
                for i in inside:
                    retained = inside[inside != i]
                    old_pair = corr[i, retained].sum()
                    for j in outside:
                        delta = unary[j] - unary[i] - config.correlation_penalty * (corr[j, retained].sum() - old_pair)
                        if delta > best_delta + 1e-12:
                            best_delta, move = float(delta), (i, j)
                if move is None:
                    break
                trial[move[0]] = 0
                trial[move[1]] = 1
                value += best_delta
            if value > best_value:
                bits, best_value = trial.copy(), value
    chosen_idx = np.flatnonzero(bits)
    selected_corr = corr[np.ix_(chosen_idx, chosen_idx)]
    pair_count = k * (k - 1)
    mean_abs_corr = float(selected_corr.sum() / pair_count) if pair_count else 0.0
    return {
        "tickers": sorted(tickers[i] for i in chosen_idx),
        "objective": reduction_objective(bits, unary, corr, config.correlation_penalty),
        "mean_abs_correlation": mean_abs_corr,
    }


def ewma_covariance(return_panel: pd.DataFrame, tickers: list[str], decision_time: pd.Timestamp, span: int, shrinkage: float) -> np.ndarray:
    panel = return_panel.loc[
        (return_panel.index <= decision_time) & (return_panel.index >= decision_time - pd.Timedelta(days=max(span * 4, 252))),
        tickers,
    ].tail(max(span * 3, 60)).fillna(0.0)
    if len(panel) < 20:
        return np.eye(len(tickers)) * 1e-4
    decay = 2.0 / (span + 1.0)
    weights = (1.0 - decay) ** np.arange(len(panel) - 1, -1, -1)
    weights /= weights.sum()
    values = panel.to_numpy(float)
    centered = values - weights @ values
    cov = (centered * weights[:, None]).T @ centered
    cov = (cov + cov.T) / 2
    if shrinkage > 0:
        target = np.diag(np.diag(cov))
        cov = (1 - shrinkage) * cov + shrinkage * target
    return cov + np.eye(len(tickers)) * 1e-8


def exact_cardinality_qubo(mu: np.ndarray, cov: np.ndarray, k: int, risk_aversion: float):
    mu_scale = max(float(np.max(np.abs(mu))), 1e-9)
    cov_scale = max(float(np.max(np.abs(cov))), 1e-9)
    q = risk_aversion * cov / cov_scale - np.diag(mu / mu_scale)
    states = np.zeros((math.comb(len(mu), k), len(mu)), dtype=np.int8)
    for row, combo in enumerate(combinations(range(len(mu)), k)):
        states[row, list(combo)] = 1
    energies = np.einsum("bi,ij,bj->b", states, q, states)
    return states[int(np.argmin(energies))], q


def project_bounded_simplex(values: np.ndarray, lower: float, upper: float, target: float = 1.0) -> np.ndarray:
    """Euclidean projection onto {w: sum(w)=target, lower<=w_i<=upper}.

    Clipping followed by normalisation is not sufficient because normalisation
    can reintroduce an upper-bound violation.  The Lagrange multiplier for this
    convex projection is found by monotone bisection.
    """
    values = np.asarray(values, dtype=float)
    n = len(values)
    if n == 0:
        return values.copy()
    tolerance = 1e-12
    if n * lower > target + tolerance or n * upper < target - tolerance:
        raise ValueError(
            f"Infeasible weight bounds: n={n}, lower={lower}, upper={upper}, target={target}"
        )
    lo = float(np.min(values - upper))
    hi = float(np.max(values - lower))
    for _ in range(100):
        multiplier = 0.5 * (lo + hi)
        projected = np.clip(values - multiplier, lower, upper)
        if projected.sum() > target:
            lo = multiplier
        else:
            hi = multiplier
    projected = np.clip(values - 0.5 * (lo + hi), lower, upper)
    # Bisection is already machine-accurate; distribute any final roundoff only
    # among coordinates that are not sitting on a bound.
    residual = target - float(projected.sum())
    free = (projected > lower + tolerance) & (projected < upper - tolerance)
    if abs(residual) > tolerance and free.any():
        projected[free] += residual / int(free.sum())
    return projected


def optimize_weights(mu: np.ndarray, cov: np.ndarray, previous: np.ndarray, config: StrategyConfig) -> np.ndarray:
    n = len(mu)
    if config.weight_mode == "equal":
        weights = np.ones(n) / n
    elif config.weight_mode == "inverse_volatility":
        inv = 1.0 / np.sqrt(np.maximum(np.diag(cov), 1e-10))
        weights = inv / inv.sum()
    else:
        if config.weight_mode == "normalized_mean_variance":
            mu_used = (mu - mu.mean()) / max(float(mu.std(ddof=0)), 1e-8)
            cov_used = cov / max(float(np.trace(cov) / n), 1e-10)
        else:
            mu_used, cov_used = mu, cov

        def objective(w):
            turnover = np.sqrt((w - previous) ** 2 + 1e-8).sum()
            return float(
                config.risk_aversion_weights * w @ cov_used @ w
                - mu_used @ w
                + config.turnover_penalty * turnover
            )

        result = minimize(
            objective,
            np.ones(n) / n,
            method="SLSQP",
            bounds=[(config.weight_lower, config.weight_upper)] * n,
            constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1.0}],
            options={"maxiter": 500, "ftol": 1e-10},
        )
        weights = np.asarray(result.x if result.success else np.ones(n) / n, float)

    weights = project_bounded_simplex(weights, config.weight_lower, config.weight_upper)
    if config.volatility_target > 0:
        annual_vol = float(np.sqrt(max(weights @ cov @ weights, 0.0) * 252))
        scale = min(1.0, config.volatility_target / max(annual_vol, 1e-9))
        weights *= scale
    return weights


def portfolio_turnover(previous: dict[str, float], target: dict[str, float]) -> float:
    names = set(previous) | set(target)
    return 0.5 * sum(abs(target.get(name, 0.0) - previous.get(name, 0.0)) for name in names)


def make_configs() -> list[StrategyConfig]:
    configs = [StrategyConfig("B00_current", "baseline", 8, 4, 0.40, 0.05, "raw_mean_variance")]
    constraint_sets = [
        ("C1", 8, 4, 0.30, 0.05),
        ("C2", 10, 6, 0.25, 0.02),
        ("C3", 10, 8, 0.15, 0.02),
    ]
    weight_sets = [
        ("EW", "equal", 1.25, 0.0, 0.0),
        ("IV", "inverse_volatility", 1.25, 0.0, 0.0),
        ("NMV", "normalized_mean_variance", 1.0, 0.10, 0.0),
        ("NMVT", "normalized_mean_variance", 2.0, 0.20, 0.15),
    ]
    for constraint_id, k, kp, upper, lower in constraint_sets:
        for weight_id, mode, risk, shrinkage, turnover_penalty in weight_sets:
            for blend_label, blend in (("X", 1.0), ("M", 0.70)):
                config_id = f"{constraint_id}_{weight_id}_{blend_label}"
                configs.append(StrategyConfig(
                    config_id, "constraint_and_allocation", k, kp, upper, lower, mode,
                    signal_blend=blend,
                    covariance_shrinkage=shrinkage,
                    risk_aversion_weights=risk,
                    turnover_penalty=turnover_penalty,
                ))
    # Focused sensitivity variants: stronger pairwise redundancy and persistence.
    for penalty in (0.20, 0.30):
        for stability in (0.15, 0.30):
            configs.append(StrategyConfig(
                f"R_K10P6_CP{int(penalty*100):02d}_S{int(stability*100):02d}",
                "reduction_sensitivity", 10, 6, 0.25, 0.02, "normalized_mean_variance",
                signal_blend=0.70, correlation_penalty=penalty, stability_weight=stability,
                covariance_shrinkage=0.20, risk_aversion_weights=2.0, turnover_penalty=0.15,
            ))
    # Phase-2 hybrid QAUR variants. Stability is set to zero so H1 compares the
    # two search mechanisms under an identical fold-level unary objective; the
    # warm start guarantees QAUR never starts below the AUR feasible solution.
    for k, kp, upper in ((8, 4, 0.30), (10, 6, 0.25)):
        for penalty in (0.30, 0.50, 0.75):
            configs.append(StrategyConfig(
                f"W_K{k}P{kp}_CP{int(penalty*100):02d}",
                "warm_started_qaur", k, kp, upper, 0.02,
                "normalized_mean_variance",
                signal_blend=0.70,
                correlation_penalty=penalty,
                stability_weight=0.0,
                covariance_shrinkage=0.20,
                risk_aversion_weights=2.0,
                turnover_penalty=0.15,
                qa_warm_start=True,
            ))
    # Common risk overlays are tested only after the reduction/constraint grid.
    # They are identical for AUR and QAUR and can move the portfolio to cash.
    overlay_specs = [
        ("NONE", 0, -1.0, 0.0),
        ("IC0", 0, 0.0, 0.0),
        ("M60", 60, -1.0, 0.0),
        ("M120", 120, -1.0, 0.0),
        ("M200", 200, -1.0, 0.0),
        ("IC0_M120", 120, 0.0, 0.0),
        ("M120_VT15", 120, -1.0, 0.15),
        ("IC0_M120_VT15", 120, 0.0, 0.15),
    ]
    for label, lookback, minimum_ic, vol_target in overlay_specs:
        configs.append(StrategyConfig(
            f"P_K10P6_CP30_{label}",
            "common_risk_overlay", 10, 6, 0.25, 0.02,
            "normalized_mean_variance",
            signal_blend=0.70,
            correlation_penalty=0.30,
            stability_weight=0.0,
            covariance_shrinkage=0.20,
            risk_aversion_weights=2.0,
            turnover_penalty=0.15,
            volatility_target=vol_target,
            qa_warm_start=True,
            market_regime_lookback=lookback,
            minimum_validation_ic=minimum_ic,
        ))
    return configs


def run_configuration(config: StrategyConfig, snapshots: pd.DataFrame, return_panel: pd.DataFrame, folds: list[dict], qa_seed: int = SEED):
    return_rows: list[dict] = []
    fold_rows: list[dict] = []
    selection_rows: list[dict] = []
    previous_universe = {"AUR": set(), "QAUR": set()}
    previous_weights: dict[str, dict[str, float]] = {"AUR": {}, "QAUR": {}}
    for fold in folds:
        fold_id = fold["fold"]
        snapshot = snapshots[snapshots["fold"].eq(fold_id)].copy()
        decision_time = pd.Timestamp(snapshot["decision_time"].iloc[0])
        risk_on = True
        if config.market_regime_lookback > 0:
            market_proxy = return_panel.mean(axis=1).loc[:decision_time].tail(config.market_regime_lookback)
            market_growth = float((1 + market_proxy.fillna(0.0)).prod() - 1)
            risk_on = risk_on and market_growth > 0
        if "validation_rank_ic" in snapshot and config.minimum_validation_ic > -1:
            validation_ic = float(snapshot["validation_rank_ic"].iloc[0])
            risk_on = risk_on and validation_ic >= config.minimum_validation_ic
        reduced: dict[str, dict] = {}
        for method in ("AUR", "QAUR"):
            reduced[method] = reduce_universe(
                method, snapshot, return_panel, previous_universe[method], config, qa_seed + fold_id,
            )
            candidates = reduced[method]["tickers"]
            candidate_snapshot = snapshot.set_index("ticker").reindex(candidates)
            signal = config.signal_blend * candidate_snapshot["xgb_signal"] + (1 - config.signal_blend) * candidate_snapshot["momentum_signal"]
            mu = signal.to_numpy(float)
            cov = ewma_covariance(return_panel, candidates, decision_time, config.covariance_span, config.covariance_shrinkage)
            bits, _ = exact_cardinality_qubo(mu, cov, config.portfolio_cardinality, config.risk_aversion_qubo)
            chosen_idx = np.flatnonzero(bits)
            chosen = [candidates[i] for i in chosen_idx]
            chosen_cov = cov[np.ix_(chosen_idx, chosen_idx)]
            previous_vector = np.array([previous_weights[method].get(t, 0.0) for t in chosen])
            weights = optimize_weights(mu[chosen_idx], chosen_cov, previous_vector, config)
            if not risk_on:
                weights = np.zeros_like(weights)
            target = dict(zip(chosen, weights))
            turnover = portfolio_turnover(previous_weights[method], target)
            test = return_panel.loc[
                (return_panel.index >= fold["test_start"]) & (return_panel.index < fold["test_end"]), chosen,
            ].fillna(0.0)
            daily = test.to_numpy(float) @ weights
            if len(daily):
                daily[0] -= turnover * config.transaction_cost_bps / 10000.0
            for date, value in zip(test.index, daily):
                return_rows.append({
                    "config_id": config.config_id, "fold": fold_id, "date": date,
                    "method": method, "return": float(value),
                })
            candidate_turnover = 1.0 - len(set(candidates) & previous_universe[method]) / config.candidate_size if previous_universe[method] else 1.0
            fold_rows.append({
                "config_id": config.config_id, "fold": fold_id, "method": method,
                "reduction_objective": reduced[method]["objective"],
                "candidate_mean_abs_correlation": reduced[method]["mean_abs_correlation"],
                "candidate_turnover": candidate_turnover,
                "portfolio_turnover": turnover,
                "risk_on": risk_on,
            })
            for ticker in candidates:
                selection_rows.append({
                    "config_id": config.config_id, "fold": fold_id, "method": method,
                    "ticker": ticker, "selected_downstream": ticker in chosen,
                    "weight": float(target.get(ticker, 0.0)),
                })
            previous_universe[method] = set(candidates)
            previous_weights[method] = target

        a_set, q_set = set(reduced["AUR"]["tickers"]), set(reduced["QAUR"]["tickers"])
        for row in fold_rows[-2:]:
            row["candidate_jaccard"] = len(a_set & q_set) / len(a_set | q_set)
    return pd.DataFrame(return_rows), pd.DataFrame(fold_rows), pd.DataFrame(selection_rows)


def summarize_configuration(config: StrategyConfig, returns: pd.DataFrame, folds: list[dict], development_last_fold: int):
    rows: list[dict] = []
    for sample, fold_filter in (
        ("development", lambda x: x <= development_last_fold),
        ("holdout", lambda x: x > development_last_fold),
        ("all", lambda x: np.ones(len(x), dtype=bool)),
    ):
        data = returns[fold_filter(returns["fold"])]
        for method, group in data.groupby("method"):
            metrics = financial_metrics(group.sort_values("date")["return"])
            rows.append({"config_id": config.config_id, "sample": sample, "method": method, **metrics})
    return rows


def xy_qaoa_statevector_audit(q: np.ndarray, k: int, seed: int, depth: int = 2, budget: int = 30, shots: int = 1024):
    states = np.zeros((math.comb(len(q), k), len(q)), dtype=np.int8)
    for row, combo in enumerate(combinations(range(len(q)), k)):
        states[row, list(combo)] = 1
    costs = np.einsum("bi,ij,bj->b", states, q, states)
    dimension = len(states)
    mixer = np.zeros((dimension, dimension))
    for i in range(dimension):
        distance = np.abs(states[i + 1:] - states[i]).sum(axis=1)
        neighbors = np.flatnonzero(distance == 2) + i + 1
        mixer[i, neighbors] = 1.0
        mixer[neighbors, i] = 1.0
    eigvals, eigvecs = np.linalg.eigh(mixer)
    initial = np.ones(dimension, dtype=complex) / np.sqrt(dimension)
    scaled_costs = costs / max(float(np.max(np.abs(costs))), 1e-12)

    def evaluate(parameters):
        psi = initial.copy()
        for gamma, beta in zip(parameters[:depth], parameters[depth:]):
            psi *= np.exp(-1j * gamma * scaled_costs)
            coeff = eigvecs.T.conj() @ psi
            psi = eigvecs @ (np.exp(-1j * beta * eigvals) * coeff)
        probabilities = np.abs(psi) ** 2
        probabilities /= probabilities.sum()
        return float(probabilities @ costs), probabilities

    rng = np.random.default_rng(seed)
    best = None
    for _ in range(3):
        x0 = np.r_[rng.uniform(0, 2 * np.pi, depth), rng.uniform(0, np.pi, depth)]
        result = minimize(lambda p: evaluate(p)[0], x0, method="COBYLA", options={"maxiter": max(8, budget // 3)})
        expected, probabilities = evaluate(result.x)
        if best is None or expected < best[0]:
            best = expected, probabilities
    sampled = rng.choice(dimension, size=shots, p=best[1])
    observed = np.unique(sampled)
    best_observed = observed[int(np.argmin(costs[observed]))]
    exact_index = int(np.argmin(costs))
    return {
        "feasibility_rate": 1.0,
        "optimality_gap": float((costs[best_observed] - costs[exact_index]) / max(abs(costs[exact_index]), 1e-12)),
        "success_probability": float(best[1][np.isclose(costs, costs.min())].sum()),
    }


def paired_tests(best_returns: pd.DataFrame, best_folds: pd.DataFrame, development_last_fold: int) -> pd.DataFrame:
    holdout_returns = best_returns[best_returns["fold"] > development_last_fold]
    wide = holdout_returns.pivot(index="date", columns="method", values="return").dropna()
    difference = wide["QAUR"] - wide["AUR"]
    t = stats.ttest_rel(wide["QAUR"], wide["AUR"], alternative="greater")
    fold_holdout = best_folds[best_folds["fold"] > development_last_fold]
    pivot = lambda column: fold_holdout.pivot(index="fold", columns="method", values=column).dropna()
    objective = pivot("reduction_objective")
    correlation = pivot("candidate_mean_abs_correlation")
    turnover = pivot("candidate_turnover")
    objective_test = stats.ttest_rel(objective["QAUR"], objective["AUR"], alternative="greater")
    correlation_test = stats.ttest_rel(correlation["QAUR"], correlation["AUR"], alternative="less")
    noninferiority_margin = 0.02
    turnover_diff = turnover["QAUR"] - turnover["AUR"]
    standard_error = float(turnover_diff.std(ddof=1) / np.sqrt(len(turnover_diff)))
    if standard_error > 0:
        statistic = float((turnover_diff.mean() - noninferiority_margin) / standard_error)
        noninferiority_p = float(stats.t.cdf(statistic, len(turnover_diff) - 1))
    else:
        statistic = -np.inf if turnover_diff.mean() < noninferiority_margin else np.inf
        noninferiority_p = 0.0 if turnover_diff.mean() < noninferiority_margin else 1.0
    return pd.DataFrame([
        {"hypothesis": "H1_QAUR_higher_QUR_objective", "estimate": float((objective["QAUR"] - objective["AUR"]).mean()), "statistic": objective_test.statistic, "pvalue_one_sided": objective_test.pvalue, "supported_5pct": objective_test.pvalue < 0.05},
        {"hypothesis": "H2_QAUR_lower_candidate_correlation", "estimate": float((correlation["QAUR"] - correlation["AUR"]).mean()), "statistic": correlation_test.statistic, "pvalue_one_sided": correlation_test.pvalue, "supported_5pct": correlation_test.pvalue < 0.05},
        {"hypothesis": "H3_QAUR_turnover_noninferior_margin_2pp", "estimate": float(turnover_diff.mean()), "statistic": statistic, "pvalue_one_sided": noninferiority_p, "supported_5pct": noninferiority_p < 0.05},
        {"hypothesis": "H4_QAUR_higher_mean_daily_return", "estimate": float(difference.mean()), "statistic": t.statistic, "pvalue_one_sided": t.pvalue, "supported_5pct": t.pvalue < 0.05},
    ])


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--max-configs", type=int, default=0)
    parser.add_argument("--config-prefix", type=str, default="")
    parser.add_argument("--snapshot-cache-dir", type=Path)
    args = parser.parse_args()
    output_dir = args.output.resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()

    prices, security, benchmark = load_market_data(args.dataset)
    print(f"loaded {len(prices):,} price rows and {prices['ticker'].nunique()} tickers", flush=True)
    features = build_features(prices)
    folds = make_folds(features["date"])
    development_last_fold = int(math.floor(len(folds) * 0.68)) - 1
    print(f"folds={len(folds)}; development=0..{development_last_fold}; holdout={development_last_fold + 1}..{len(folds)-1}", flush=True)
    if args.snapshot_cache_dir:
        cache_dir = args.snapshot_cache_dir.resolve()
        snapshots = pd.read_csv(cache_dir / "forecast_snapshots.csv", parse_dates=["decision_time"])
        forecast_diagnostics = pd.read_csv(cache_dir / "forecast_diagnostics.csv", parse_dates=["decision_time"])
        snapshots.to_csv(output_dir / "forecast_snapshots.csv", index=False)
        forecast_diagnostics.to_csv(output_dir / "forecast_diagnostics.csv", index=False)
        print(f"reused forecast cache from {cache_dir}", flush=True)
    else:
        snapshots, forecast_diagnostics = build_fold_cache(features, security, folds, output_dir)
    snapshots = snapshots.drop(columns=["validation_rank_ic"], errors="ignore").merge(
        forecast_diagnostics[["fold", "validation_rank_ic"]], on="fold", how="left",
    )
    return_panel = features.pivot(index="date", columns="ticker", values="return_1d").sort_index()

    configs = make_configs()
    if args.config_prefix:
        configs = [config for config in configs if config.config_id.startswith(args.config_prefix)]
    if args.max_configs > 0:
        configs = configs[:args.max_configs]
    if not configs:
        raise ValueError("No configurations matched the requested filter.")
    pd.DataFrame([asdict(config) for config in configs]).to_csv(output_dir / "configuration_definitions.csv", index=False)
    all_summaries: list[dict] = []
    all_fold_diagnostics: list[pd.DataFrame] = []
    config_returns: dict[str, pd.DataFrame] = {}
    config_selections: dict[str, pd.DataFrame] = {}
    for index, config in enumerate(configs, start=1):
        config_started = time.perf_counter()
        returns, diagnostics, selections = run_configuration(config, snapshots, return_panel, folds)
        all_summaries.extend(summarize_configuration(config, returns, folds, development_last_fold))
        all_fold_diagnostics.append(diagnostics)
        config_returns[config.config_id] = returns
        config_selections[config.config_id] = selections
        print(f"config {index:02d}/{len(configs)} {config.config_id} done in {time.perf_counter()-config_started:.1f}s", flush=True)

    summary = pd.DataFrame(all_summaries)
    pd.concat(config_returns.values(), ignore_index=True).to_csv(
        output_dir / "all_configuration_returns.csv", index=False,
    )
    pd.concat(config_selections.values(), ignore_index=True).to_csv(
        output_dir / "all_configuration_selections.csv", index=False,
    )
    development = summary[summary["sample"].eq("development")].pivot(index="config_id", columns="method", values="sharpe_zero_rf")
    development["selection_score"] = development[["AUR", "QAUR"]].mean(axis=1)
    best_id = str(development["selection_score"].idxmax())
    best_config = next(config for config in configs if config.config_id == best_id)
    summary = summary.merge(development[["selection_score"]], left_on="config_id", right_index=True, how="left")
    summary.to_csv(output_dir / "configuration_results.csv", index=False)
    fold_diagnostics = pd.concat(all_fold_diagnostics, ignore_index=True)
    fold_diagnostics.to_csv(output_dir / "all_fold_diagnostics.csv", index=False)

    best_returns = config_returns[best_id]
    best_selections = config_selections[best_id]
    best_folds = fold_diagnostics[fold_diagnostics["config_id"].eq(best_id)].copy()
    best_returns.to_csv(output_dir / "best_configuration_returns.csv", index=False)
    best_selections.to_csv(output_dir / "best_configuration_selections.csv", index=False)
    best_folds.to_csv(output_dir / "best_configuration_fold_diagnostics.csv", index=False)
    tests = paired_tests(best_returns, best_folds, development_last_fold)

    # H5: seed/configuration robustness.  Re-run the selected configuration under
    # three independent QAUR initialisation seeds, then record the return sign.
    robustness_rows: list[dict] = []
    for qa_seed in (7, 42, 99):
        returns, diagnostics, _ = run_configuration(best_config, snapshots, return_panel, folds, qa_seed=qa_seed)
        holdout = returns[returns["fold"] > development_last_fold]
        metrics = {m: financial_metrics(g.sort_values("date")["return"]) for m, g in holdout.groupby("method")}
        robustness_rows.append({
            "qa_seed": qa_seed,
            "aur_sharpe": metrics["AUR"]["sharpe_zero_rf"],
            "qaur_sharpe": metrics["QAUR"]["sharpe_zero_rf"],
            "qaur_minus_aur_sharpe": metrics["QAUR"]["sharpe_zero_rf"] - metrics["AUR"]["sharpe_zero_rf"],
            "mean_objective_gap": float(diagnostics.pivot(index="fold", columns="method", values="reduction_objective").diff(axis=1)["QAUR"].mean()),
        })
    robustness = pd.DataFrame(robustness_rows)
    robustness.to_csv(output_dir / "seed_robustness.csv", index=False)
    h5_supported = bool((robustness["qaur_minus_aur_sharpe"] > 0).all())
    tests = pd.concat([tests, pd.DataFrame([{
        "hypothesis": "H5_direction_robust_across_QAUR_seeds",
        "estimate": float((robustness["qaur_minus_aur_sharpe"] > 0).mean()),
        "statistic": np.nan,
        "pvalue_one_sided": np.nan,
        "supported_5pct": h5_supported,
    }])], ignore_index=True)
    tests.to_csv(output_dir / "hypothesis_tests.csv", index=False)

    # Final statevector audit on all holdout folds and both reducers.  It checks
    # that the screening solution remains reachable under shared XY-QAOA.
    qaoa_rows: list[dict] = []
    for fold in folds:
        if fold["fold"] <= development_last_fold:
            continue
        snapshot = snapshots[snapshots["fold"].eq(fold["fold"])].copy()
        decision_time = pd.Timestamp(snapshot["decision_time"].iloc[0])
        for method in ("AUR", "QAUR"):
            selected = best_selections[
                (best_selections["fold"].eq(fold["fold"])) & (best_selections["method"].eq(method))
            ]["ticker"].tolist()
            candidate_snapshot = snapshot.set_index("ticker").reindex(selected)
            mu = (best_config.signal_blend * candidate_snapshot["xgb_signal"] + (1-best_config.signal_blend) * candidate_snapshot["momentum_signal"]).to_numpy(float)
            cov = ewma_covariance(return_panel, selected, decision_time, best_config.covariance_span, best_config.covariance_shrinkage)
            _, q = exact_cardinality_qubo(mu, cov, best_config.portfolio_cardinality, best_config.risk_aversion_qubo)
            audit = xy_qaoa_statevector_audit(q, best_config.portfolio_cardinality, SEED + fold["fold"])
            qaoa_rows.append({"fold": fold["fold"], "method": method, **audit})
    qaoa_audit = pd.DataFrame(qaoa_rows)
    qaoa_audit.to_csv(output_dir / "xy_qaoa_holdout_audit.csv", index=False)

    # Add benchmark and full-universe EW on exactly the final holdout dates.
    holdout_start = folds[development_last_fold + 1]["test_start"]
    holdout_end = folds[-1]["test_end"]
    benchmark_holdout = benchmark.set_index("date")["return"].loc[lambda x: (x.index >= holdout_start) & (x.index < holdout_end)].dropna()
    full_ew = return_panel.loc[(return_panel.index >= holdout_start) & (return_panel.index < holdout_end)].mean(axis=1)
    benchmark_summary = pd.DataFrame([
        {"method": "FULL_UNIVERSE_EW", **financial_metrics(full_ew)},
        {"method": "VNALLSHARE_TRI", **financial_metrics(benchmark_holdout)},
    ])
    benchmark_summary.to_csv(output_dir / "holdout_baselines.csv", index=False)

    holdout_best = summary[(summary["config_id"].eq(best_id)) & (summary["sample"].eq("holdout"))]
    support_count = int(tests["supported_5pct"].fillna(False).sum())
    conclusion = f"""# Kết luận strategy search có temporal holdout

## Thiết kế

- Dữ liệu: `{args.dataset.name}`; SHA-256 `{sha256_file(args.dataset)}`.
- {len(folds)} walk-forward folds; development folds 0–{development_last_fold}, untouched holdout folds {development_last_fold + 1}–{len(folds)-1}.
- Đã sàng lọc {len(configs)} cấu hình. Cấu hình được chọn bằng Sharpe trung bình của AUR và QAUR trên development, không nhìn holdout.
- Grid dùng exact feasible-subspace reference; cấu hình thắng được audit lại bằng shared fixed-Hamming-weight XY-QAOA statevector trên holdout.

## Cấu hình đề xuất

`{best_id}`

```json
{json.dumps(asdict(best_config), indent=2, ensure_ascii=False)}
```

## Kết quả holdout

{holdout_best.to_markdown(index=False)}

## Baseline holdout

{benchmark_summary.to_markdown(index=False)}

## Giả thuyết

{tests.to_markdown(index=False)}

Có {support_count}/5 giả thuyết đạt tiêu chí đã định trước. H5 là robustness direction check, không phải kiểm định quantum advantage.

## Diễn giải hợp lệ

Kết quả dương trên development không được xem là bằng chứng nếu không lặp lại trên temporal holdout. QAUR vẫn là classical surrogate cho quantum-ready QUBO; XY-QAOA là ideal statevector simulation. Không có tuyên bố quantum advantage.
"""
    (output_dir / "research_conclusion.md").write_text(conclusion, encoding="utf-8")

    equity = (1 + best_returns.pivot(index="date", columns="method", values="return").fillna(0.0)).cumprod()
    ax = equity.plot(figsize=(11, 5), title=f"Best configuration: {best_id}")
    ax.axvline(pd.Timestamp(holdout_start), color="black", linestyle="--", label="holdout start")
    ax.set_ylabel("Growth of 1")
    ax.legend()
    plt.tight_layout()
    plt.savefig(output_dir / "best_equity_curve.png", dpi=180)
    plt.close()

    manifest = {
        "dataset_sha256": sha256_file(args.dataset),
        "folds": len(folds),
        "development_last_fold": development_last_fold,
        "holdout_first_fold": development_last_fold + 1,
        "configurations_screened": len(configs),
        "selection_rule": "maximum mean development Sharpe across AUR and QAUR",
        "best_config_id": best_id,
        "best_config": asdict(best_config),
        "runtime_seconds": time.perf_counter() - started,
        "python": platform.python_version(),
        "xgboost": __import__("xgboost").__version__,
        "qa_backend_disclosure": "classical multi-start cardinality-preserving swap surrogate",
        "screening_backend": "exact fixed-cardinality QUBO enumeration",
        "final_audit_backend": "ideal fixed-Hamming-weight XY-QAOA statevector",
        "quantum_advantage_claimed": False,
    }
    (output_dir / "run_manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"BEST_CONFIG={best_id}", flush=True)
    print(holdout_best.to_string(index=False), flush=True)
    print(tests.to_string(index=False), flush=True)
    print(f"completed in {manifest['runtime_seconds']:.1f}s; output={output_dir}", flush=True)


if __name__ == "__main__":
    main()


## 3. Mã điều phối confirmatory và practical experiments

Lớp confirmatory sàng lọc 43 cấu hình trên development, khóa cấu hình rồi mới
đánh giá 15 historical holdout folds. Lớp practical đánh giá 24 cấu hình nhân
với bốn market gates, tương ứng 96 phương án. Quy tắc practical là lợi nhuận
dương ở cả sáu ô giai đoạn–reducer, maximum drawdown không vượt 20% về độ lớn,
sau đó tối đa hóa worst-case Sharpe.


In [ ]:
%%writefile /content/aur_qaur_data_nckh_5_9/run_colab_data_nckh_5_9_complete.py
from __future__ import annotations

"""Complete 29/8 experiment orchestrator used locally and in Google Colab.

The module deliberately reports two evidence layers:

1. confirmatory historical evidence: configuration selection on folds 0--28 and
   testing on untouched folds 29--43;
2. practical method-design evidence: broader constraint/allocation/gate search
   using all information observed by 29 August 2026.  This layer may select a
   paper-trading protocol, but it is never relabelled as prospective evidence.

All AUR/QAUR comparisons use the same downstream portfolio QUBO, exact
fixed-cardinality screening reference, weight allocator, transaction costs and
market gate.  QAUR remains a classical surrogate for a quantum-ready QUBO.
"""

import argparse
from dataclasses import asdict
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

from run_constraint_strategy_search import (
    SEED,
    StrategyConfig,
    build_features,
    build_fold_cache,
    exact_cardinality_qubo,
    ewma_covariance,
    financial_metrics,
    load_market_data,
    make_configs,
    make_folds,
    paired_tests,
    run_configuration,
    xy_qaoa_statevector_audit,
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def observed_sample(fold: int) -> str:
    if fold <= 28:
        return "development_2022_2024"
    if fold <= 43:
        return "historical_holdout_2024_2025"
    if fold == 44:
        return "bridge_december_2025"
    return "observed_2026"


def add_august_and_prospective_folds(complete: list[dict]) -> list[dict]:
    previous = complete[-1]
    august = {
        key: (value + pd.DateOffset(months=1) if key != "fold" else int(value) + 1)
        for key, value in previous.items()
    }
    september = {
        key: (value + pd.DateOffset(months=1) if key != "fold" else int(value) + 1)
        for key, value in august.items()
    }
    if august["test_start"] != pd.Timestamp("2026-08-02"):
        raise RuntimeError(f"Unexpected August fold: {august}")
    if september["test_start"] != pd.Timestamp("2026-09-02"):
        raise RuntimeError(f"Unexpected September fold: {september}")
    return complete + [august, september]


def apply_market_gate(
    returns: pd.DataFrame,
    folds: list[dict],
    market: pd.Series,
    lookback: int,
    switching_cost_bps: float,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Apply one common, causal gate to both reducer branches."""
    fold_map = {int(fold["fold"]): fold for fold in folds}
    output: list[pd.DataFrame] = []
    exposure_rows: list[dict] = []
    previous_exposure = 0.0
    for fold_id in sorted(returns["fold"].unique()):
        if int(fold_id) not in fold_map:
            continue
        fold = fold_map[int(fold_id)]
        decision_time = pd.Timestamp(fold["test_start"]) - pd.Timedelta(days=1)
        if lookback <= 0:
            growth, exposure = np.nan, 1.0
        else:
            trailing = market.loc[market.index <= decision_time].tail(lookback).fillna(0.0)
            growth = float((1.0 + trailing).prod() - 1.0)
            exposure = float(growth > 0.0)
        chunk = returns[returns["fold"].eq(fold_id)].copy()
        chunk["return"] *= exposure
        if len(chunk) and exposure != previous_exposure:
            first_date = chunk["date"].min()
            first_rows = chunk["date"].eq(first_date)
            chunk.loc[first_rows, "return"] -= switching_cost_bps / 10000.0
        chunk["market_gate_exposure"] = exposure
        chunk["market_gate_growth"] = growth
        chunk["market_gate_lookback"] = lookback
        output.append(chunk)
        exposure_rows.append({
            "fold": int(fold_id),
            "decision_time": decision_time,
            "market_growth": growth,
            "exposure": exposure,
            "lookback": lookback,
        })
        previous_exposure = exposure
    return pd.concat(output, ignore_index=True), pd.DataFrame(exposure_rows)


def summarize_periods(returns: pd.DataFrame, config_id: str, gate: int) -> list[dict]:
    tagged = returns.copy()
    tagged["sample"] = tagged["fold"].astype(int).map(observed_sample)
    rows: list[dict] = []
    for (sample, method), group in tagged.groupby(["sample", "method"]):
        if sample == "bridge_december_2025":
            continue
        rows.append({
            "config_id": config_id,
            "market_gate_lookback": gate,
            "sample": sample,
            "method": method,
            **financial_metrics(group.sort_values("date")["return"]),
        })
    return rows


def robust_candidate_ranking(summary: pd.DataFrame) -> pd.DataFrame:
    grouped = summary.groupby(["config_id", "market_gate_lookback"])
    ranking = grouped.agg(
        worst_return=("cumulative_return", "min"),
        worst_sharpe=("sharpe_zero_rf", "min"),
        mean_sharpe=("sharpe_zero_rf", "mean"),
        worst_drawdown=("maximum_drawdown", "min"),
        positive_cells=("cumulative_return", lambda x: int((x > 0).sum())),
        total_cells=("cumulative_return", "size"),
    ).reset_index()
    ranking["passes_positive_all"] = ranking["positive_cells"].eq(ranking["total_cells"])
    ranking["passes_drawdown_20pct"] = ranking["worst_drawdown"].ge(-0.20)
    ranking["practical_gate_passed"] = (
        ranking["passes_positive_all"] & ranking["passes_drawdown_20pct"]
    )
    ranking = ranking.sort_values(
        ["practical_gate_passed", "worst_sharpe", "worst_return", "mean_sharpe"],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)
    return ranking


def block_bootstrap_mean_difference(
    difference: pd.Series,
    seed: int = 20260829,
    block_length: int = 20,
    repetitions: int = 5000,
) -> dict:
    values = pd.Series(difference).dropna().to_numpy(float)
    if len(values) < 2:
        return {"estimate": np.nan, "ci_low": np.nan, "ci_high": np.nan, "pvalue": np.nan}
    rng = np.random.default_rng(seed)
    starts = np.arange(max(1, len(values) - block_length + 1))
    boot = np.empty(repetitions)
    for b in range(repetitions):
        pieces: list[np.ndarray] = []
        while sum(len(piece) for piece in pieces) < len(values):
            start = int(rng.choice(starts))
            pieces.append(values[start:start + block_length])
        boot[b] = np.concatenate(pieces)[:len(values)].mean()
    return {
        "estimate": float(values.mean()),
        "ci_low": float(np.quantile(boot, 0.025)),
        "ci_high": float(np.quantile(boot, 0.975)),
        "pvalue": float((np.sum(boot <= 0.0) + 1) / (repetitions + 1)),
    }


def practical_h4_by_period(returns: pd.DataFrame) -> pd.DataFrame:
    tagged = returns.copy()
    tagged["sample"] = tagged["fold"].astype(int).map(observed_sample)
    rows: list[dict] = []
    for sample, sample_returns in tagged.groupby("sample"):
        if sample == "bridge_december_2025":
            continue
        wide = sample_returns.pivot(index="date", columns="method", values="return").dropna()
        difference = wide["QAUR"] - wide["AUR"]
        test = stats.ttest_1samp(difference, 0.0, alternative="greater")
        bootstrap = block_bootstrap_mean_difference(difference)
        rows.append({
            "sample": sample,
            "observations": len(difference),
            "mean_daily_difference": float(difference.mean()),
            "paired_t_statistic": float(test.statistic),
            "paired_t_pvalue_one_sided": float(test.pvalue),
            "block_bootstrap_ci_low": bootstrap["ci_low"],
            "block_bootstrap_ci_high": bootstrap["ci_high"],
            "block_bootstrap_pvalue_one_sided": bootstrap["pvalue"],
            "supported_5pct": bool(test.pvalue < 0.05 and bootstrap["pvalue"] < 0.05),
            "evidence_label": "posthoc_method_design_not_confirmatory",
        })
    return pd.DataFrame(rows)


def holm_adjust(pvalues: pd.Series) -> np.ndarray:
    """Holm family-wise-error adjustment without an extra dependency."""
    values = pd.Series(pvalues, dtype=float).to_numpy()
    order = np.argsort(values)
    adjusted = np.empty(len(values), dtype=float)
    running = 0.0
    for rank, index in enumerate(order):
        running = max(running, (len(values) - rank) * values[index])
        adjusted[index] = min(running, 1.0)
    return adjusted


def practical_positive_return_evidence(returns: pd.DataFrame) -> pd.DataFrame:
    """Separate positive realised P&L from statistical evidence of mean > 0."""
    tagged = returns.copy()
    tagged["sample"] = tagged["fold"].astype(int).map(observed_sample)
    rows: list[dict] = []
    for (sample, method), group in tagged.groupby(["sample", "method"]):
        if sample == "bridge_december_2025":
            continue
        daily = group.sort_values("date")["return"].dropna()
        test = stats.ttest_1samp(daily, 0.0, alternative="greater")
        bootstrap = block_bootstrap_mean_difference(daily)
        rows.append({
            "sample": sample,
            "method": method,
            "observations": len(daily),
            "mean_daily_return": float(daily.mean()),
            "cumulative_return": float((1.0 + daily).prod() - 1.0),
            "one_sample_t_pvalue": float(test.pvalue),
            "block_bootstrap_ci_low": bootstrap["ci_low"],
            "block_bootstrap_ci_high": bootstrap["ci_high"],
            "block_bootstrap_pvalue": bootstrap["pvalue"],
        })
    table = pd.DataFrame(rows)
    table["combined_conservative_pvalue"] = table[
        ["one_sample_t_pvalue", "block_bootstrap_pvalue"]
    ].max(axis=1)
    table["holm_adjusted_pvalue"] = holm_adjust(
        table["combined_conservative_pvalue"]
    )
    table["positive_economically"] = table["cumulative_return"].gt(0.0)
    table["positive_mean_supported_holm_5pct"] = table[
        "holm_adjusted_pvalue"
    ].lt(0.05)
    table["evidence_label"] = "posthoc_method_design_not_confirmatory"
    return table


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--snapshot-cache", type=Path)
    parser.add_argument("--max-practical-configs", type=int, default=0)
    args = parser.parse_args()

    dataset = args.dataset.resolve()
    output = args.output.resolve()
    output.mkdir(parents=True, exist_ok=True)
    forecast_dir = output / "forecast_cache"
    forecast_dir.mkdir(exist_ok=True)

    prices, security, benchmark = load_market_data(dataset)
    features = build_features(prices)
    complete_folds = make_folds(features["date"])
    all_folds = add_august_and_prospective_folds(complete_folds)
    observed_folds = all_folds[:-1]
    if args.snapshot_cache:
        cache = args.snapshot_cache.resolve()
        snapshots = pd.read_csv(cache / "forecast_snapshots.csv", parse_dates=["decision_time"])
        forecast_diagnostics = pd.read_csv(cache / "forecast_diagnostics.csv", parse_dates=["decision_time"])
    else:
        snapshots, forecast_diagnostics = build_fold_cache(
            features, security, all_folds, forecast_dir
        )
    snapshots = snapshots.drop(columns=["validation_rank_ic"], errors="ignore").merge(
        forecast_diagnostics[["fold", "validation_rank_ic"]], on="fold", how="left"
    )
    return_panel = features.pivot(index="date", columns="ticker", values="return_1d").sort_index()
    market = return_panel.mean(axis=1).sort_index()

    # Confirmatory layer: exactly the 2020--2025 experiment and original split.
    historical_folds = [fold for fold in complete_folds if int(fold["fold"]) <= 43]
    confirmatory_configs = make_configs()
    confirmatory_returns: dict[str, pd.DataFrame] = {}
    confirmatory_diagnostics: list[pd.DataFrame] = []
    confirmatory_selections: dict[str, pd.DataFrame] = {}
    confirmatory_summary: list[dict] = []
    for index, config in enumerate(confirmatory_configs, 1):
        returns, diagnostics, selections = run_configuration(
            config, snapshots, return_panel, historical_folds
        )
        confirmatory_returns[config.config_id] = returns
        confirmatory_selections[config.config_id] = selections
        confirmatory_diagnostics.append(diagnostics)
        for sample, keep in (
            ("development", returns["fold"].le(28)),
            ("holdout", returns["fold"].between(29, 43)),
        ):
            for method, group in returns[keep].groupby("method"):
                confirmatory_summary.append({
                    "config_id": config.config_id,
                    "sample": sample,
                    "method": method,
                    **financial_metrics(group.sort_values("date")["return"]),
                })
        print(f"confirmatory {index:02d}/{len(confirmatory_configs)} {config.config_id}", flush=True)
    confirmatory_summary_table = pd.DataFrame(confirmatory_summary)
    development = confirmatory_summary_table[
        confirmatory_summary_table["sample"].eq("development")
    ].pivot(index="config_id", columns="method", values="sharpe_zero_rf")
    development["selection_score"] = development[["AUR", "QAUR"]].mean(axis=1)
    confirmatory_best_id = str(development["selection_score"].idxmax())
    confirmatory_fold_table = pd.concat(confirmatory_diagnostics, ignore_index=True)
    confirmatory_best_folds = confirmatory_fold_table[
        confirmatory_fold_table["config_id"].eq(confirmatory_best_id)
    ]
    confirmatory_tests = paired_tests(
        confirmatory_returns[confirmatory_best_id], confirmatory_best_folds, 28
    )
    confirmatory_best_config = next(
        config for config in confirmatory_configs if config.config_id == confirmatory_best_id
    )
    confirmatory_best_returns = confirmatory_returns[confirmatory_best_id].copy()
    confirmatory_best_selections = confirmatory_selections[confirmatory_best_id].copy()

    # Classical baselines are evaluated on exactly the same untouched holdout
    # dates as the locked AUR/QAUR configuration.  These rows are kept separate
    # from H4 because the confirmatory paired test remains QAUR minus AUR.
    holdout_start = historical_folds[29]["test_start"]
    holdout_end = historical_folds[-1]["test_end"]
    full_universe_ew = return_panel.loc[
        (return_panel.index >= holdout_start) & (return_panel.index < holdout_end)
    ].mean(axis=1)
    benchmark_holdout = benchmark.set_index("date")["return"].loc[
        lambda values: (values.index >= holdout_start) & (values.index < holdout_end)
    ].dropna()
    confirmatory_holdout_baselines = pd.DataFrame([
        {"method": "FULL_UNIVERSE_EW", **financial_metrics(full_universe_ew)},
        {"method": "VNALLSHARE_TRI", **financial_metrics(benchmark_holdout)},
    ])
    confirmatory_seed_rows: list[dict] = []
    for qa_seed in (7, 42, 99):
        seed_returns, seed_diagnostics, _ = run_configuration(
            confirmatory_best_config, snapshots, return_panel, historical_folds,
            qa_seed=qa_seed,
        )
        holdout = seed_returns[seed_returns["fold"].between(29, 43)]
        seed_perf = {
            method: financial_metrics(group.sort_values("date")["return"])
            for method, group in holdout.groupby("method")
        }
        objective = seed_diagnostics.pivot(
            index="fold", columns="method", values="reduction_objective"
        ).dropna()
        confirmatory_seed_rows.append({
            "qa_seed": qa_seed,
            "aur_sharpe": seed_perf["AUR"]["sharpe_zero_rf"],
            "qaur_sharpe": seed_perf["QAUR"]["sharpe_zero_rf"],
            "qaur_minus_aur_sharpe": (
                seed_perf["QAUR"]["sharpe_zero_rf"]
                - seed_perf["AUR"]["sharpe_zero_rf"]
            ),
            "mean_qaur_objective_advantage": float(
                (objective["QAUR"] - objective["AUR"]).mean()
            ),
        })
    confirmatory_seed_robustness = pd.DataFrame(confirmatory_seed_rows)
    h5_supported = bool(
        (confirmatory_seed_robustness["qaur_minus_aur_sharpe"] > 0).all()
    )
    confirmatory_tests = pd.concat([
        confirmatory_tests,
        pd.DataFrame([{
            "hypothesis": "H5_QAUR_financial_direction_robust_across_seeds",
            "estimate": float(
                (confirmatory_seed_robustness["qaur_minus_aur_sharpe"] > 0).mean()
            ),
            "statistic": np.nan,
            "pvalue_one_sided": np.nan,
            "supported_5pct": h5_supported,
        }]),
    ], ignore_index=True)
    confirmatory_tests["holm_adjusted_pvalue"] = np.nan
    inferential = confirmatory_tests["pvalue_one_sided"].notna()
    confirmatory_tests.loc[inferential, "holm_adjusted_pvalue"] = holm_adjust(
        confirmatory_tests.loc[inferential, "pvalue_one_sided"]
    )
    confirmatory_tests["supported_holm_5pct"] = (
        confirmatory_tests["holm_adjusted_pvalue"].lt(0.05)
    )
    confirmatory_tests["evidence_label"] = "confirmatory_untouched_historical_holdout"

    # Shared downstream XY-QAOA audit.  The same circuit depth, optimisation
    # budget and number of shots are used for the AUR and QAUR candidate sets.
    # The exact fixed-cardinality solution remains the reference used to
    # calculate success probability and optimality gap.
    confirmatory_xy_rows: list[dict] = []
    for fold in historical_folds:
        if int(fold["fold"]) <= 28:
            continue
        fold_id = int(fold["fold"])
        snapshot = snapshots[snapshots["fold"].eq(fold_id)].copy()
        decision_time = pd.Timestamp(snapshot["decision_time"].iloc[0])
        for method in ("AUR", "QAUR"):
            candidates = confirmatory_selections[confirmatory_best_id].loc[
                lambda frame: frame["fold"].eq(fold_id)
                & frame["method"].eq(method),
                "ticker",
            ].tolist()
            candidate_snapshot = snapshot.set_index("ticker").reindex(candidates)
            mu = (
                confirmatory_best_config.signal_blend
                * candidate_snapshot["xgb_signal"]
                + (1.0 - confirmatory_best_config.signal_blend)
                * candidate_snapshot["momentum_signal"]
            ).to_numpy(float)
            cov = ewma_covariance(
                return_panel,
                candidates,
                decision_time,
                confirmatory_best_config.covariance_span,
                confirmatory_best_config.covariance_shrinkage,
            )
            _, q_matrix = exact_cardinality_qubo(
                mu,
                cov,
                confirmatory_best_config.portfolio_cardinality,
                confirmatory_best_config.risk_aversion_qubo,
            )
            audit = xy_qaoa_statevector_audit(
                q_matrix,
                confirmatory_best_config.portfolio_cardinality,
                SEED + fold_id,
            )
            confirmatory_xy_rows.append({
                "fold": fold_id,
                "method": method,
                "candidate_size": len(candidates),
                "portfolio_cardinality": confirmatory_best_config.portfolio_cardinality,
                "depth": 2,
                "budget": 30,
                "shots": 1024,
                **audit,
            })
    confirmatory_xy_audit = pd.DataFrame(confirmatory_xy_rows)

    # Practical method-design layer: constraint/allocation families plus a
    # common market gate.  This uses observed 2026 and is explicitly post-hoc.
    practical_configs = [
        config for config in make_configs()
        if config.family == "constraint_and_allocation"
    ]
    if args.max_practical_configs > 0:
        practical_configs = practical_configs[:args.max_practical_configs]
    practical_return_map: dict[tuple[str, int], pd.DataFrame] = {}
    practical_selection_map: dict[str, pd.DataFrame] = {}
    practical_diagnostic_map: dict[str, pd.DataFrame] = {}
    practical_summary_rows: list[dict] = []
    exposure_parts: list[pd.DataFrame] = []
    for index, config in enumerate(practical_configs, 1):
        base_returns, diagnostics, selections = run_configuration(
            config, snapshots, return_panel, observed_folds
        )
        practical_selection_map[config.config_id] = selections
        practical_diagnostic_map[config.config_id] = diagnostics
        for gate in (0, 20, 30, 40):
            gated, exposures = apply_market_gate(
                base_returns, observed_folds, market, gate,
                switching_cost_bps=25.0 if gate > 0 else 0.0,
            )
            practical_return_map[(config.config_id, gate)] = gated
            practical_summary_rows.extend(summarize_periods(gated, config.config_id, gate))
            exposures["config_id"] = config.config_id
            exposure_parts.append(exposures)
        print(f"practical {index:02d}/{len(practical_configs)} {config.config_id}", flush=True)

    practical_summary = pd.DataFrame(practical_summary_rows)
    practical_ranking = robust_candidate_ranking(practical_summary)
    if not practical_ranking["practical_gate_passed"].any():
        raise RuntimeError("No practical configuration passed positive-return and drawdown gates.")
    selected_row = practical_ranking[practical_ranking["practical_gate_passed"]].iloc[0]
    practical_best_id = str(selected_row["config_id"])
    practical_best_gate = int(selected_row["market_gate_lookback"])
    practical_best_config = next(c for c in practical_configs if c.config_id == practical_best_id)
    practical_best_returns = practical_return_map[(practical_best_id, practical_best_gate)]
    exploratory_h4 = practical_h4_by_period(practical_best_returns)
    positive_return_evidence = practical_positive_return_evidence(
        practical_best_returns
    )

    # Seed robustness for the selected practical base configuration.
    seed_rows: list[dict] = []
    for qa_seed in (7, 29, 101, 1009, 20260829):
        base_returns, diagnostics, _ = run_configuration(
            practical_best_config, snapshots, return_panel, observed_folds, qa_seed=qa_seed
        )
        gated, _ = apply_market_gate(
            base_returns, observed_folds, market, practical_best_gate,
            25.0 if practical_best_gate > 0 else 0.0,
        )
        observed = gated[gated["fold"].ge(45)]
        perf = {
            method: financial_metrics(group.sort_values("date")["return"])
            for method, group in observed.groupby("method")
        }
        pivot = diagnostics.pivot(index="fold", columns="method", values="reduction_objective").dropna()
        seed_rows.append({
            "qa_seed": qa_seed,
            "aur_sharpe": perf["AUR"]["sharpe_zero_rf"],
            "qaur_sharpe": perf["QAUR"]["sharpe_zero_rf"],
            "qaur_minus_aur_sharpe": perf["QAUR"]["sharpe_zero_rf"] - perf["AUR"]["sharpe_zero_rf"],
            "mean_qaur_objective_advantage": float((pivot["QAUR"] - pivot["AUR"]).mean()),
        })
    seed_robustness = pd.DataFrame(seed_rows)

    # Generate the September shadow basket using all past states, then apply the
    # selected common gate.  The prospective fold has no future returns yet.
    _, prospective_diagnostics, prospective_selections = run_configuration(
        practical_best_config, snapshots, return_panel, all_folds
    )
    prospective_fold = int(all_folds[-1]["fold"])
    shadow = prospective_selections[
        prospective_selections["fold"].eq(prospective_fold)
        & prospective_selections["selected_downstream"]
    ].copy()
    decision_time = pd.Timestamp(snapshots[snapshots["fold"].eq(prospective_fold)]["decision_time"].iloc[0])
    if practical_best_gate > 0:
        trailing = market.loc[market.index <= decision_time].tail(practical_best_gate).fillna(0.0)
        current_growth = float((1 + trailing).prod() - 1)
        current_exposure = float(current_growth > 0)
    else:
        current_growth, current_exposure = np.nan, 1.0
    shadow["shadow_weight"] = shadow["weight"]
    shadow["executable_weight"] = shadow["shadow_weight"] * current_exposure
    shadow["cash_weight"] = 1.0 - current_exposure
    shadow["market_growth"] = current_growth
    shadow["market_gate_exposure"] = current_exposure

    # Preserve the complete prospective Top-K sets, not only the four selected
    # assets.  The explanatory Colab cells merge these rows with the live
    # forecast/risk snapshot so that the final basket is auditable from source
    # signals rather than entered manually.
    prospective_candidates = prospective_selections[
        prospective_selections["fold"].eq(prospective_fold)
    ].copy()
    prospective_snapshot = snapshots[snapshots["fold"].eq(prospective_fold)][[
        "ticker", "decision_time", "xgb_signal", "momentum_signal",
        "liquidity_20d", "volatility_20d",
    ]].copy()
    prospective_candidates = prospective_candidates.merge(
        prospective_snapshot, on="ticker", how="left", validate="many_to_one"
    )
    prospective_candidates["shadow_weight"] = prospective_candidates["weight"]
    prospective_candidates["executable_weight"] = (
        prospective_candidates["shadow_weight"] * current_exposure
    )
    prospective_candidates["cash_weight"] = 1.0 - current_exposure
    prospective_candidates["market_growth"] = current_growth
    prospective_candidates["market_gate_exposure"] = current_exposure

    # Export every table required for paper reporting and reproducibility.
    confirmatory_summary_table.to_csv(output / "confirmatory_configuration_results.csv", index=False)
    confirmatory_tests.to_csv(output / "confirmatory_hypothesis_tests.csv", index=False)
    confirmatory_seed_robustness.to_csv(
        output / "confirmatory_seed_robustness.csv", index=False
    )
    confirmatory_xy_audit.to_csv(
        output / "confirmatory_xy_qaoa_holdout_audit.csv", index=False
    )
    confirmatory_best_returns.to_csv(
        output / "confirmatory_best_returns.csv", index=False
    )
    confirmatory_best_folds.to_csv(
        output / "confirmatory_best_fold_diagnostics.csv", index=False
    )
    confirmatory_best_selections.to_csv(
        output / "confirmatory_best_selections.csv", index=False
    )
    confirmatory_holdout_baselines.to_csv(
        output / "confirmatory_holdout_baselines.csv", index=False
    )
    practical_summary.to_csv(output / "practical_configuration_period_results.csv", index=False)
    practical_ranking.to_csv(output / "practical_robust_ranking.csv", index=False)
    practical_best_returns.to_csv(output / "selected_practical_returns.csv", index=False)
    exploratory_h4.to_csv(output / "selected_practical_h4_by_period.csv", index=False)
    positive_return_evidence.to_csv(
        output / "selected_practical_positive_return_evidence.csv", index=False
    )
    seed_robustness.to_csv(output / "selected_practical_seed_robustness.csv", index=False)
    practical_diagnostic_map[practical_best_id].to_csv(
        output / "selected_practical_fold_diagnostics.csv", index=False
    )
    practical_selection_map[practical_best_id].to_csv(
        output / "selected_practical_selections.csv", index=False
    )
    pd.concat(exposure_parts, ignore_index=True).to_csv(output / "market_gate_exposures.csv", index=False)
    shadow.to_csv(output / "september_2026_shadow_and_executable_basket.csv", index=False)
    prospective_candidates.to_csv(
        output / "september_2026_candidate_audit.csv", index=False
    )
    forecast_diagnostics.to_csv(output / "forecast_diagnostics.csv", index=False)
    snapshots.to_csv(output / "forecast_snapshots.csv", index=False)

    selected_periods = practical_summary[
        practical_summary["config_id"].eq(practical_best_id)
        & practical_summary["market_gate_lookback"].eq(practical_best_gate)
    ].copy()
    selected_periods.to_csv(output / "selected_practical_period_results.csv", index=False)
    pd.DataFrame(all_folds).to_csv(output / "walk_forward_fold_manifest.csv", index=False)
    pd.DataFrame([asdict(config) for config in confirmatory_configs]).to_csv(
        output / "confirmatory_configuration_definitions.csv", index=False
    )
    pd.DataFrame([asdict(config) for config in practical_configs]).to_csv(
        output / "practical_configuration_definitions.csv", index=False
    )
    manifest = {
        "dataset": dataset.name,
        "dataset_sha256": sha256_file(dataset),
        "observed_price_end": str(features["date"].max().date()),
        "confirmatory_fold_split": {"development": "0-28", "holdout": "29-43"},
        "confirmatory_configurations_screened": len(confirmatory_configs),
        "confirmatory_best_config": confirmatory_best_id,
        "practical_evidence_label": "posthoc_method_design_using_data_observed_by_2026_08_29",
        "practical_configurations_screened": len(practical_configs),
        "market_gate_lookbacks_screened": [0, 20, 30, 40],
        "practical_best_config": asdict(practical_best_config),
        "practical_best_market_gate_lookback": practical_best_gate,
        "practical_selection_rule": "positive return in all 3 periods x 2 reducers; MDD >= -20%; maximize worst Sharpe",
        "prospective_protocol_start": "2026-09-02",
        "current_market_growth": current_growth,
        "current_exposure": current_exposure,
        "live_capital_authorized": False,
        "quantum_advantage_claimed": False,
        "qa_backend": "classical cardinality-preserving surrogate for quantum-ready QUBO",
        "shared_downstream_screening": "exact fixed-cardinality QUBO reference",
        "xy_qaoa_holdout_audit_instances": int(len(confirmatory_xy_audit)),
        "xy_qaoa_mean_feasibility_rate": float(
            confirmatory_xy_audit["feasibility_rate"].mean()
        ),
    }
    (output / "run_manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
    )

    report = f"""# Kết quả tối ưu hóa thực tiễn — bản dữ liệu 29/8

## Lớp xác nhận lịch sử

- Cấu hình chọn chỉ từ development: `{confirmatory_best_id}`.
- Holdout: folds 29--43.

{confirmatory_tests.to_markdown(index=False)}

XY-QAOA holdout audit ({len(confirmatory_xy_audit)} instances): mean feasibility
rate = {confirmatory_xy_audit["feasibility_rate"].mean():.4f}, mean optimality gap
= {confirmatory_xy_audit["optimality_gap"].mean():.6f}.

## Lớp thiết kế phương pháp thực tiễn

- Cấu hình: `{practical_best_id}`.
- Common market gate: {practical_best_gate} phiên.
- Nhãn bằng chứng: **post-hoc method design**, chưa phải prospective proof.

{selected_periods.to_markdown(index=False)}

## H4 theo từng giai đoạn

{exploratory_h4.to_markdown(index=False)}

## Lợi nhuận dương: hiệu quả kinh tế và ý nghĩa thống kê

{positive_return_evidence.to_markdown(index=False)}

## Rổ tháng 9/2026

{shadow[["method", "ticker", "shadow_weight", "executable_weight", "cash_weight", "market_growth"]].to_markdown(index=False)}

## Kết luận hợp lệ

Phương pháp được phép chuyển sang paper trading không vốn từ 02/09/2026 nếu giữ
nguyên tham số. Không có kết quả nào trong lớp practical được diễn giải thành
quantum advantage hoặc cho phép triển khai vốn thật.
"""
    (output / "FINAL_RESULTS_29_8_VI.md").write_text(report, encoding="utf-8")
    print(json.dumps(manifest, indent=2, ensure_ascii=False), flush=True)
    print(selected_periods.to_string(index=False), flush=True)
    print(shadow[["method", "ticker", "shadow_weight", "executable_weight"]].to_string(index=False), flush=True)


if __name__ == "__main__":
    main()


## 4. Chạy toàn bộ hệ thống

Cell này fit lại XGBoost theo từng fold, tạo cả hai candidate sets, chạy shared
portfolio pipeline, backtest, statistical tests và final-basket decision. Thời
gian chạy Colab CPU dự kiến khoảng 20–30 phút. Các bảng trung gian chỉ tồn tại
trong runtime để mô-đun hóa pipeline; mọi kết quả nghiên cứu được hiển thị trực
tiếp bên dưới và không có bước tải file.


In [ ]:

import subprocess, sys, time

RESULTS = WORKDIR / "results_data_nckh_5_9"
RESULTS.mkdir(exist_ok=True)
started = time.time()
command = [
    sys.executable,
    str(WORKDIR / "run_colab_data_nckh_5_9_complete.py"),
    "--dataset", str(DATASET),
    "--output", str(RESULTS),
]
print("Running:", " ".join(command))
subprocess.run(command, check=True, cwd=WORKDIR)
print(f"Hoàn tất sau {(time.time()-started)/60:.1f} phút")


In [ ]:

import json

def read_result(name, **kwargs):
    return pd.read_csv(RESULTS / name, **kwargs)

manifest = json.loads((RESULTS / "run_manifest.json").read_text(encoding="utf-8"))
fold_manifest = read_result("walk_forward_fold_manifest.csv", parse_dates=[
    "train_start", "train_end", "validation_start", "validation_end",
    "test_start", "test_end",
])
forecast_diagnostics = read_result("forecast_diagnostics.csv", parse_dates=["decision_time"])
forecast_snapshots = read_result("forecast_snapshots.csv", parse_dates=["decision_time"])
confirmatory_definitions = read_result("confirmatory_configuration_definitions.csv")
practical_definitions = read_result("practical_configuration_definitions.csv")
confirmatory_configurations = read_result("confirmatory_configuration_results.csv")
confirmatory_tests = read_result("confirmatory_hypothesis_tests.csv")
confirmatory_seeds = read_result("confirmatory_seed_robustness.csv")
confirmatory_folds = read_result("confirmatory_best_fold_diagnostics.csv")
confirmatory_selections = read_result("confirmatory_best_selections.csv")
confirmatory_returns = read_result("confirmatory_best_returns.csv", parse_dates=["date"])
confirmatory_baselines = read_result("confirmatory_holdout_baselines.csv")
xy_audit = read_result("confirmatory_xy_qaoa_holdout_audit.csv")
practical_all = read_result("practical_configuration_period_results.csv")
practical_ranking = read_result("practical_robust_ranking.csv")
selected_periods = read_result("selected_practical_period_results.csv")
selected_returns = read_result("selected_practical_returns.csv", parse_dates=["date"])
selected_folds = read_result("selected_practical_fold_diagnostics.csv")
selected_selections = read_result("selected_practical_selections.csv")
positive_evidence = read_result("selected_practical_positive_return_evidence.csv")
practical_h4 = read_result("selected_practical_h4_by_period.csv")
practical_seeds = read_result("selected_practical_seed_robustness.csv")
market_exposures = read_result("market_gate_exposures.csv", parse_dates=["decision_time"])
final_candidates = read_result("september_2026_candidate_audit.csv", parse_dates=["decision_time"])
final_basket = read_result("september_2026_shadow_and_executable_basket.csv")

display(Markdown("### Run manifest"))
display(pd.DataFrame({"field": list(manifest), "value": [str(manifest[k]) for k in manifest]}))
display(Markdown("### Walk-forward fold manifest"))
display(fold_manifest)


## 5. Kết quả Giai đoạn 1 — XGBoost và EWMA

Rank IC đo tương quan thứ hạng giữa forecast và lợi suất quan sát. Giá trị dương
cho thấy mô hình có xu hướng xếp tài sản sinh lợi cao lên trên, nhưng không trực
tiếp chứng minh alpha có ý nghĩa thống kê. EWMA được dùng thống nhất cho cả AUR
và QAUR nên không tạo confounding effect giữa hai nhánh.


In [ ]:

def evidence_period(fold):
    fold = int(fold)
    if fold <= 28:
        return "development_0_28"
    if fold <= 43:
        return "historical_holdout_29_43"
    if fold <= 52:
        return "observed_extension_44_52"
    return "prospective_basket_53"

forecast_view = forecast_diagnostics.copy()
forecast_view["evidence_period"] = forecast_view["fold"].map(evidence_period)
forecast_summary = forecast_view.groupby("evidence_period", sort=False).agg(
    folds=("fold", "nunique"),
    mean_rank_ic=("validation_rank_ic", "mean"),
    median_rank_ic=("validation_rank_ic", "median"),
    positive_rank_ic_folds=("validation_rank_ic", lambda x: int((x > 0).sum())),
    mean_rmse=("validation_rmse", "mean"),
    mean_universe_size=("universe_size", "mean"),
    total_runtime_seconds=("runtime_seconds", "sum"),
).reset_index()

display(Markdown("### Forecast summary theo lớp bằng chứng"))
display(forecast_summary)
display(Markdown("### Chẩn đoán của toàn bộ 54 folds"))
display(forecast_view)

observed_forecast = forecast_view[forecast_view["fold"].le(52)]
mean_ic = observed_forecast["validation_rank_ic"].mean()
positive_ic = int((observed_forecast["validation_rank_ic"] > 0).sum())
display(Markdown(f'''
**Diễn giải.** Có **{len(forecast_view)} forecast snapshots**; 53 folds đầu đã
có test window quan sát, còn fold 53 tạo quyết định prospective. Trên 53 folds
đã quan sát, validation Rank IC trung bình bằng **{mean_ic:.4f}** và dương tại
**{positive_ic}/53 folds**. Kết quả cho thấy năng lực xếp hạng dương ở mức mô
tả, nhưng độ biến thiên giữa folds và sự chồng lấn cửa sổ không cho phép coi
đây là 53 bằng chứng thống kê độc lập về alpha.
'''))


## 6. Kết quả Giai đoạn 2 — Adaptive Universe Reduction

AUR sử dụng cùng unary score với QAUR nhưng xây Top-\(K\) theo cơ chế greedy
tuần tự. Bảng dưới đây hiển thị đầy đủ candidate set và tập tài sản được shared
portfolio pipeline lựa chọn tại từng historical holdout fold.


In [ ]:

locked_id = manifest["confirmatory_best_config"]
confirmatory_holdout_selections = confirmatory_selections[
    confirmatory_selections["fold"].between(29, 43)
].copy()

def selection_by_fold(frame, method):
    chosen = frame[frame["method"].eq(method)].copy()
    candidates = chosen.groupby("fold")["ticker"].agg(lambda x: ", ".join(x))
    portfolio = chosen[chosen["selected_downstream"]].groupby("fold")["ticker"].agg(lambda x: ", ".join(x))
    weights = chosen[chosen["selected_downstream"]].groupby("fold").apply(
        lambda x: ", ".join(f"{t}:{w:.2%}" for t, w in zip(x["ticker"], x["weight"])),
        include_groups=False,
    )
    return pd.concat([candidates.rename("Top-K"), portfolio.rename("selected_kp"), weights.rename("weights")], axis=1).reset_index()

aur_topk = selection_by_fold(confirmatory_holdout_selections, "AUR")
display(Markdown(f"**Cấu hình đã khóa:** `{locked_id}`"))
display(aur_topk)
aur_reduction = confirmatory_folds[
    confirmatory_folds["fold"].between(29, 43) & confirmatory_folds["method"].eq("AUR")
]
display(aur_reduction)


## 7. Kết quả Giai đoạn 3 — Quantum-Assisted Universe Reduction

QAUR tối ưu joint quality–redundancy objective trong fixed-cardinality
subspace. Đây là quantum-assisted formulation, nhưng reducer đang được giải
bằng classical cardinality-preserving search; kết quả không được diễn giải là
quantum speedup.


In [ ]:

qaur_topk = selection_by_fold(confirmatory_holdout_selections, "QAUR")
display(qaur_topk)
qaur_reduction = confirmatory_folds[
    confirmatory_folds["fold"].between(29, 43) & confirmatory_folds["method"].eq("QAUR")
]
display(qaur_reduction)


## 8. Kết quả Giai đoạn 4 — So sánh AUR và QAUR; kiểm định H1–H5

H1–H4 sử dụng paired one-sided tests trên untouched historical holdout và được
hiệu chỉnh Holm. H3 dùng non-inferiority margin 2 điểm phần trăm. H5 là điều
kiện robustness về hướng Sharpe qua các seed, không gán p-value giả tạo.


In [ ]:

holdout_fold_metrics = confirmatory_folds[confirmatory_folds["fold"].between(29, 43)]
metric_wide = holdout_fold_metrics.pivot(index="fold", columns="method", values=[
    "reduction_objective", "candidate_mean_abs_correlation", "candidate_turnover"
])
comparison_by_fold = pd.DataFrame({
    "fold": metric_wide.index,
    "QUR_AUR": metric_wide[("reduction_objective", "AUR")],
    "QUR_QAUR": metric_wide[("reduction_objective", "QAUR")],
    "correlation_AUR": metric_wide[("candidate_mean_abs_correlation", "AUR")],
    "correlation_QAUR": metric_wide[("candidate_mean_abs_correlation", "QAUR")],
    "turnover_AUR": metric_wide[("candidate_turnover", "AUR")],
    "turnover_QAUR": metric_wide[("candidate_turnover", "QAUR")],
})
comparison_by_fold["candidate_jaccard"] = holdout_fold_metrics.groupby("fold")["candidate_jaccard"].first()
display(Markdown("### So sánh theo từng holdout fold"))
display(comparison_by_fold.reset_index(drop=True))

hypothesis_text = {
    "H1_QAUR_higher_QUR_objective": "QAUR có Q^UR cao hơn AUR",
    "H2_QAUR_lower_candidate_correlation": "QAUR có candidate correlation thấp hơn AUR",
    "H3_QAUR_turnover_noninferior_margin_2pp": "QAUR không kém hơn AUR về turnover (margin 2pp)",
    "H4_QAUR_higher_mean_daily_return": "QAUR có mean daily return cao hơn AUR",
    "H5_QAUR_financial_direction_robust_across_seeds": "Ưu thế Sharpe của QAUR ổn định qua seed",
}
hypothesis_table = confirmatory_tests.copy()
hypothesis_table.insert(1, "statement", hypothesis_table["hypothesis"].map(hypothesis_text))
hypothesis_table["conclusion"] = np.where(
    hypothesis_table["supported_holm_5pct"].fillna(False),
    "Được ủng hộ", "Không được ủng hộ"
)
display(Markdown("### Bảng kiểm định giả thuyết"))
display(hypothesis_table)
display(Markdown("### Seed robustness của confirmatory configuration"))
display(confirmatory_seeds)

supported = hypothesis_table.loc[hypothesis_table["conclusion"].eq("Được ủng hộ"), "hypothesis"].str[:2].tolist()
unsupported = hypothesis_table.loc[hypothesis_table["conclusion"].ne("Được ủng hộ"), "hypothesis"].str[:2].tolist()
mean_jaccard = comparison_by_fold["candidate_jaccard"].mean()
display(Markdown(f'''
**Kết luận thống kê.** Candidate-set Jaccard trung bình bằng
**{mean_jaccard:.4f}**. Các giả thuyết được ủng hộ là **{', '.join(supported)}**;
các giả thuyết chưa được ủng hộ là **{', '.join(unsupported)}**. Vì vậy, bằng
chứng xác nhận ưu thế của QAUR nằm ở tầng universe reduction; chưa có bằng
chứng rằng ưu thế này chuyển thành mean daily return hoặc Sharpe cao hơn.
'''))


## 9. Kết quả Giai đoạn 5 — Shared Portfolio QUBO và XY-QAOA

Feasibility rate kiểm tra tỷ lệ samples thỏa \(\sum_i x_i=k_p\). Optimality gap
so sánh nghiệm tốt nhất quan sát với exact fixed-cardinality reference. Success
probability là xác suất single-shot của nghiệm tối ưu, không phải xác suất sinh
lợi của danh mục.


In [ ]:

display(Markdown("### Toàn bộ 30 XY-QAOA holdout audit instances"))
display(xy_audit)
xy_summary = xy_audit.groupby("method").agg(
    instances=("fold", "size"),
    feasibility_rate=("feasibility_rate", "mean"),
    mean_optimality_gap=("optimality_gap", "mean"),
    mean_success_probability=("success_probability", "mean"),
    min_success_probability=("success_probability", "min"),
).reset_index()
display(Markdown("### Tổng hợp solver audit"))
display(xy_summary)
display(Markdown("### Sáu tài sản được chọn từ mỗi Top-10 tại từng holdout fold"))
selected_holdout = confirmatory_holdout_selections[
    confirmatory_holdout_selections["selected_downstream"]
].pivot_table(index="fold", columns="method", values="ticker", aggfunc=lambda x: ", ".join(x)).reset_index()
display(selected_holdout)

mean_feasible = xy_audit["feasibility_rate"].mean()
mean_gap = xy_audit["optimality_gap"].mean()
mean_success = xy_audit["success_probability"].mean()
display(Markdown(f'''
**Diễn giải.** Feasibility trung bình đạt **{mean_feasible:.2%}**, mean
optimality gap bằng **{mean_gap:.6f}** và single-shot success probability trung
bình bằng **{mean_success:.2%}**. Kết quả chứng minh formulation giữ đúng
cardinality và có thể quan sát nghiệm exact reference trên các instances nhỏ
với 1.024 shots. Nó không chứng minh quantum advantage vì statevector simulator
chạy trên phần cứng cổ điển và exact enumeration vẫn khả thi với candidate set
nhỏ.
'''))


## 10. Kết quả Giai đoạn 6 — Walk-Forward Backtest ngoài mẫu

Các metrics được tính trên return ròng sau transaction-cost assumptions.
Maximum drawdown là mức giảm sâu nhất của tài sản từ một đỉnh lịch sử xuống đáy
tiếp theo; ví dụ −15% nghĩa là tại thời điểm tệ nhất, giá trị danh mục thấp hơn
đỉnh gần nhất 15%.


In [ ]:

confirmatory_holdout = confirmatory_configurations[
    confirmatory_configurations["config_id"].eq(locked_id)
    & confirmatory_configurations["sample"].eq("holdout")
].drop(columns=["config_id", "sample"])
confirmatory_performance = pd.concat([
    confirmatory_holdout,
    confirmatory_baselines,
], ignore_index=True)
confirmatory_performance["final_wealth"] = 1.0 + confirmatory_performance["cumulative_return"]
confirmatory_performance["calmar_zero_rf"] = (
    confirmatory_performance["annualized_return"]
    / confirmatory_performance["maximum_drawdown"].abs().replace(0, np.nan)
)
display(Markdown("### Untouched historical holdout: reducers và baselines"))
display(confirmatory_performance)

best_return_method = confirmatory_performance.loc[
    confirmatory_performance["cumulative_return"].idxmax(), "method"
]
best_drawdown_method = confirmatory_performance.loc[
    confirmatory_performance["maximum_drawdown"].idxmax(), "method"
]
h4_row = hypothesis_table[hypothesis_table["hypothesis"].str.startswith("H4")].iloc[0]
display(Markdown(f'''
**Diễn giải.** Phương pháp có cumulative return cao nhất trên holdout là
**{best_return_method}**; phương pháp có maximum drawdown nông nhất là
**{best_drawdown_method}**. Chênh lệch QAUR–AUR về mean daily return bằng
**{h4_row['estimate']:.8f}**, với Holm-adjusted p-value
**{h4_row['holm_adjusted_pvalue']:.4f}**. Do đó, khác biệt tài chính giữa hai
reducer không có ý nghĩa thống kê ở mức 5%. Candidate set tốt hơn chưa chắc tạo
lợi nhuận cao hơn vì shared Q^PO chỉ giữ k_p tài sản và classical optimizer tiếp
tục làm tương đồng risk exposures.
'''))


## 11. Practical method design, ràng buộc và độ bền lợi nhuận

Phần này là post-hoc method design. Mục tiêu là tìm cấu hình có hiệu quả kinh tế
ổn định trong không gian đã khai báo, không thay thế confirmatory evidence.


In [ ]:

selected_id = manifest["practical_best_config"]["config_id"]
selected_gate = int(manifest["practical_best_market_gate_lookback"])
selected_config = pd.DataFrame([manifest["practical_best_config"]])
selected_periods_view = selected_periods.copy()
selected_periods_view["final_wealth"] = 1.0 + selected_periods_view["cumulative_return"]
selected_periods_view["calmar_zero_rf"] = (
    selected_periods_view["annualized_return"]
    / selected_periods_view["maximum_drawdown"].abs().replace(0, np.nan)
)

selected_exposures = market_exposures[
    market_exposures["config_id"].eq(selected_id)
    & market_exposures["lookback"].eq(selected_gate)
].sort_values("fold")
exposure_days = selected_returns.groupby("method").agg(
    observations=("date", "size"),
    invested_days=("market_gate_exposure", lambda x: int((x > 0).sum())),
    cash_days=("market_gate_exposure", lambda x: int((x == 0).sum())),
).reset_index()
transaction_cost_bps = float(manifest["practical_best_config"]["transaction_cost_bps"])
cost_summary = selected_folds.groupby("method").agg(
    total_one_way_turnover=("portfolio_turnover", "sum"),
    mean_rebalance_turnover=("portfolio_turnover", "mean"),
).reset_index()
cost_summary["modeled_rebalance_cost_rate"] = (
    cost_summary["total_one_way_turnover"] * transaction_cost_bps / 10000.0
)

display(Markdown("### Cấu hình practical được lựa chọn"))
display(selected_config.T.rename(columns={0: "value"}))
display(Markdown(f"**Common market gate:** {selected_gate} phiên"))
display(Markdown("### Kết quả theo ba giai đoạn và hai reducer"))
display(selected_periods_view)
display(Markdown("### Số ngày có exposure và giữ tiền mặt"))
display(exposure_days)
display(Markdown("### Turnover và chi phí mô phỏng"))
display(cost_summary)
display(Markdown("### Bằng chứng lợi nhuận dương"))
display(positive_evidence)
display(Markdown("### H4 exploratory theo từng giai đoạn"))
display(practical_h4)
display(Markdown("### Practical seed robustness"))
display(practical_seeds)
display(Markdown("### Top 10 trong xếp hạng 96 phương án"))
display(practical_ranking.head(10))

economic_positive = int(positive_evidence["positive_economically"].sum())
statistical_positive = int(positive_evidence["positive_mean_supported_holm_5pct"].sum())
worst_sharpe = selected_periods["sharpe_zero_rf"].min()
worst_drawdown = selected_periods["maximum_drawdown"].min()
display(Markdown(f'''
**Diễn giải.** Cấu hình `{selected_id} + gate {selected_gate}` có lợi nhuận
quan sát dương tại **{economic_positive}/6** ô, worst-case Sharpe bằng
**{worst_sharpe:.4f}** và maximum drawdown tệ nhất bằng
**{worst_drawdown:.2%}**. Tuy nhiên, chỉ **{statistical_positive}/6** ô có mean
daily return dương sau Holm ở mức 5%. Vì cấu hình được chọn sau khi quan sát 96
phương án, kết quả này là post-hoc và chỉ đủ làm cơ sở cho paper trading.
'''))


## 12. Final Stock Basket and Execution Status

Top-K cuối cùng và rổ bốn cổ phiếu bên dưới được pipeline tính trực tiếp từ
fold prospective; không có mã cổ phiếu hoặc tỷ trọng nào được nhập thủ công.
Signal/liquidity/risk ranks chỉ giải thích vị trí tương đối trong candidate set,
không phải khuyến nghị mua bán.


In [ ]:

final_candidates = final_candidates.copy()
final_candidates["signal_rank_in_candidate"] = final_candidates.groupby("method")["xgb_signal"].rank(pct=True)
final_candidates["liquidity_rank_in_candidate"] = final_candidates.groupby("method")["liquidity_20d"].rank(pct=True)
final_candidates["low_risk_rank_in_candidate"] = final_candidates.groupby("method")["volatility_20d"].rank(pct=True, ascending=False)
final_candidates = final_candidates.sort_values(["method", "selected_downstream", "shadow_weight"], ascending=[True, False, False])

display(Markdown("### Top-K_A và Top-K_QA, kèm quyết định shared portfolio pipeline"))
display(final_candidates[[
    "decision_time", "method", "ticker", "selected_downstream", "xgb_signal",
    "momentum_signal", "liquidity_20d", "volatility_20d",
    "signal_rank_in_candidate", "liquidity_rank_in_candidate",
    "low_risk_rank_in_candidate", "shadow_weight", "executable_weight",
]])

final_selected = final_candidates[final_candidates["selected_downstream"]].copy()
display(Markdown("### Rổ cổ phiếu cuối cùng"))
display(final_selected[[
    "method", "ticker", "shadow_weight", "executable_weight", "cash_weight",
    "market_growth", "market_gate_exposure",
]])

aur_set = set(final_selected.loc[final_selected["method"].eq("AUR"), "ticker"])
qaur_set = set(final_selected.loc[final_selected["method"].eq("QAUR"), "ticker"])
same_final_set = aur_set == qaur_set
gate_exposure = float(final_selected["market_gate_exposure"].iloc[0])
cash_weight = float(final_selected["cash_weight"].iloc[0])
market_growth = float(final_selected["market_growth"].iloc[0])

representative = final_selected[final_selected["method"].eq("AUR")]
asset_lines = []
for row in representative.itertuples():
    asset_lines.append(
        f"- **{row.ticker}** — shadow weight {row.shadow_weight:.2%}; "
        f"signal rank {row.signal_rank_in_candidate:.0%}, liquidity rank "
        f"{row.liquidity_rank_in_candidate:.0%}, low-risk rank "
        f"{row.low_risk_rank_in_candidate:.0%} trong Top-K."
    )

status = "được phép có exposure" if gate_exposure > 0 else "bị market gate chặn"
convergence = "hội tụ về cùng một tập tài sản" if same_final_set else "tạo hai tập tài sản cuối khác nhau"
display(Markdown(
    f'''**Giải thích rổ cuối.** AUR và QAUR **{convergence}**. Market proxy 30
phiên tăng trưởng **{market_growth:.2%}**, vì vậy quyết định hiện tại **{status}**.
Cash weight bằng **{cash_weight:.2%}**. Các shadow weights mô tả danh mục sẽ
được theo dõi khi risk-on; executable weights mới là quyết định thực thi.

''' + "\n".join(asset_lines) + "\n\nRổ này phục vụ paper trading, không phải khuyến nghị đầu tư."
))


## 13. Biểu đồ kết quả

Các biểu đồ dưới đây được tạo từ object của lần chạy hiện tại và hiển thị trực
tiếp trong notebook.


In [ ]:

import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 2, figsize=(16, 22))

# Forecast Rank IC
axes[0, 0].plot(forecast_diagnostics["fold"], forecast_diagnostics["validation_rank_ic"], marker="o", ms=3)
axes[0, 0].axhline(0, color="black", lw=1, ls="--")
axes[0, 0].set(title="Validation Rank IC by fold", xlabel="Fold", ylabel="Rank IC")

# Candidate similarity
axes[0, 1].plot(comparison_by_fold["fold"], comparison_by_fold["candidate_jaccard"], marker="o", color="#6a1b9a")
axes[0, 1].set(title="AUR–QAUR candidate-set similarity", xlabel="Holdout fold", ylabel="Jaccard")
axes[0, 1].set_ylim(0, 1.05)

# QUR and correlation
axes[1, 0].plot(comparison_by_fold["fold"], comparison_by_fold["QUR_AUR"], label="AUR")
axes[1, 0].plot(comparison_by_fold["fold"], comparison_by_fold["QUR_QAUR"], label="QAUR")
axes[1, 0].set(title="Quality–redundancy objective", xlabel="Holdout fold", ylabel="Q^UR")
axes[1, 0].legend()
axes[1, 1].plot(comparison_by_fold["fold"], comparison_by_fold["correlation_AUR"], label="AUR")
axes[1, 1].plot(comparison_by_fold["fold"], comparison_by_fold["correlation_QAUR"], label="QAUR")
axes[1, 1].set(title="Candidate mean absolute correlation", xlabel="Holdout fold", ylabel="Mean |correlation|")
axes[1, 1].legend()

# Practical cumulative wealth and drawdown
for method, group in selected_returns.groupby("method"):
    group = group.sort_values("date")
    wealth = (1 + group["return"]).cumprod()
    drawdown = wealth / wealth.cummax() - 1
    axes[2, 0].plot(group["date"], wealth, label=method)
    axes[2, 1].plot(group["date"], drawdown, label=method)
axes[2, 0].set(title="Practical cumulative wealth", xlabel="Date", ylabel="Growth of 1")
axes[2, 1].set(title="Practical drawdown", xlabel="Date", ylabel="Drawdown")
axes[2, 0].legend(); axes[2, 1].legend()

# Holdout risk-return comparison
for row in confirmatory_performance.itertuples():
    axes[3, 0].scatter(row.annualized_volatility, row.annualized_return, s=70)
    axes[3, 0].annotate(row.method, (row.annualized_volatility, row.annualized_return), xytext=(4, 4), textcoords="offset points")
axes[3, 0].set(title="Historical holdout risk–return", xlabel="Annualized volatility", ylabel="Annualized return")

# Final basket and market gate
basket_plot = representative.set_index("ticker")["shadow_weight"].sort_values(ascending=False)
axes[3, 1].bar(basket_plot.index, basket_plot.values, color="#ef6c00")
axes[3, 1].set(title=f"Final shadow basket; executable exposure={gate_exposure:.0%}", xlabel="Ticker", ylabel="Weight")

for ax in axes.flat:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()
figure_created = True

display(Markdown('''
**Cách đọc.** Hai biểu đồ đầu mô tả forecast và mức trùng lặp reducer; hai biểu
đồ tiếp theo tách quality objective khỏi correlation redundancy; equity curve
và drawdown phản ánh hiệu quả kinh tế sau chi phí; risk–return plot đặt AUR và
QAUR cạnh các baseline; biểu đồ cuối là shadow weights, không phải executable
weights khi market gate đóng.
'''))


## 14. Tổng hợp học thuật và mức độ sẵn sàng ứng dụng

Phần dưới được sinh từ kết quả của lần chạy hiện tại, không chèn sẵn số liệu.


In [ ]:

best_practical = selected_periods_view.loc[selected_periods_view["sharpe_zero_rf"].idxmax()]
supported_labels = ", ".join(supported) if supported else "không có"
unsupported_labels = ", ".join(unsupported) if unsupported else "không có"

display(Markdown(f'''
### Kết luận kết quả

Bộ dữ liệu gồm **{len(raw_data):,} bản ghi**, **{price_raw['ticker'].nunique()} mã
cổ phiếu**, bao phủ từ **{price_raw['date'].min().date()}** đến
**{price_raw['date'].max().date()}**. Pipeline tạo **{len(forecast_diagnostics)}
walk-forward snapshots**. Năng lực xếp hạng của XGBoost mang dấu dương ở mức mô
tả nhưng biến động theo fold; EWMA cung cấp cùng risk input cho hai reducer.

Ở tầng universe reduction, các giả thuyết được ủng hộ là
**{supported_labels}**; các giả thuyết chưa được ủng hộ là
**{unsupported_labels}**. Điều này xác định phạm vi ưu thế của QAUR ở candidate
quality, correlation và turnover, nhưng không cho phép suy ra ưu thế lợi nhuận.

XY-QAOA đạt feasibility **{mean_feasible:.2%}** và mean optimality gap
**{mean_gap:.6f}** trên {len(xy_audit)} instances. Đây là bằng chứng về tính
đúng đắn của feasible-subspace implementation trên simulator, không phải bằng
chứng quantum advantage.

Cấu hình practical tốt nhất là **{selected_id} + gate {selected_gate}**. Lợi
nhuận quan sát dương tại **{economic_positive}/6** ô nhưng lợi nhuận dương có ý
nghĩa thống kê sau Holm chỉ đạt **{statistical_positive}/6** ô. Maximum drawdown
tệ nhất là **{worst_drawdown:.2%}**. Do đó, kết quả kinh tế tương đối ổn định
trong mẫu đã xét nhưng bằng chứng thống kê vẫn chưa đủ mạnh.

Rổ prospective gồm **{', '.join(sorted(aur_set | qaur_set))}**. Market gate
hiện đặt exposure bằng **{gate_exposure:.0%}**, nên executable portfolio là
**{cash_weight:.0%} tiền mặt**; shadow basket chỉ được theo dõi trong paper
trading.

### Mức độ áp dụng

Framework hiện phù hợp cho **paper trading không sử dụng vốn thật**. Chưa nên
triển khai live capital vì practical selection là post-hoc, positive-return
tests chưa đạt ý nghĩa sau multiple-testing correction, dữ liệu 2026 còn mang
tính provisional và chưa có prospective record đủ dài với slippage, market
impact cùng sự cố dữ liệu thực tế. Bước tiếp theo là khóa toàn bộ tham số, chạy
forward 6–12 tháng, công bố cả kỳ giữ tiền mặt và tiếp tục so sánh với
Full-Universe equal-weight cùng VNAllShare TRI.
'''))


## 15. Research audit


In [ ]:

audit_rows = []
def audit(name, condition, evidence):
    audit_rows.append({"check": name, "passed": bool(condition), "evidence": str(evidence)})

folds_chronological = (
    (fold_manifest["train_end"] <= fold_manifest["validation_start"]).all()
    and (fold_manifest["validation_end"] <= fold_manifest["test_start"]).all()
)
locked_definition = confirmatory_definitions[confirmatory_definitions["config_id"].eq(locked_id)].iloc[0]
practical_definition = practical_definitions[practical_definitions["config_id"].eq(selected_id)].iloc[0]
confirm_candidate_counts = confirmatory_selections.groupby(["fold", "method"])["ticker"].nunique()
confirm_selected_counts = confirmatory_selections[confirmatory_selections["selected_downstream"]].groupby(["fold", "method"])["ticker"].nunique()
practical_candidate_counts = selected_selections.groupby(["fold", "method"])["ticker"].nunique()
practical_selected = selected_selections[selected_selections["selected_downstream"]]
practical_selected_counts = practical_selected.groupby(["fold", "method"])["ticker"].nunique()
practical_weight_sums = practical_selected.groupby(["fold", "method"])["weight"].sum()

audit("Dataset SHA-256", digest == manifest["dataset_sha256"], digest)
audit("No duplicate price keys", duplicate_price_keys == 0, duplicate_price_keys)
audit("Chronological train-validation-test folds", folds_chronological, len(fold_manifest))
audit("All 54 forecast snapshots present", len(forecast_diagnostics) == 54, len(forecast_diagnostics))
audit("AUR/QAUR share the same fold set", confirmatory_selections.groupby("fold")["method"].nunique().eq(2).all(), "two reducers per fold")
audit("Confirmatory Top-K cardinality", confirm_candidate_counts.eq(int(locked_definition["candidate_size"])).all(), confirm_candidate_counts.unique())
audit("Confirmatory portfolio cardinality", confirm_selected_counts.eq(int(locked_definition["portfolio_cardinality"])).all(), confirm_selected_counts.unique())
audit("Practical Top-K cardinality", practical_candidate_counts.eq(int(practical_definition["candidate_size"])).all(), practical_candidate_counts.unique())
audit("Practical portfolio cardinality", practical_selected_counts.eq(int(practical_definition["portfolio_cardinality"])).all(), practical_selected_counts.unique())
audit("Practical weights sum to one before common gate", np.allclose(practical_weight_sums, 1.0, atol=1e-7), practical_weight_sums.min())
audit("Practical weight lower bound", practical_selected["weight"].ge(float(practical_definition["weight_lower"]) - 1e-8).all(), practical_selected["weight"].min())
audit("Practical weight upper bound", practical_selected["weight"].le(float(practical_definition["weight_upper"]) + 1e-8).all(), practical_selected["weight"].max())
audit("Transaction-cost inputs are non-negative", selected_folds["portfolio_turnover"].ge(0).all() and transaction_cost_bps >= 0, transaction_cost_bps)
audit("XY-QAOA fixed-cardinality feasibility", xy_audit["feasibility_rate"].eq(1.0).all(), xy_audit["feasibility_rate"].mean())
audit("Five hypotheses reported", set(confirmatory_tests["hypothesis"]) == set(hypothesis_text), len(confirmatory_tests))
audit("Holm p-values available for H1-H4", confirmatory_tests["holm_adjusted_pvalue"].notna().sum() == 4, confirmatory_tests["holm_adjusted_pvalue"].notna().sum())
audit("Final shadow weights sum to one per reducer", final_selected.groupby("method")["shadow_weight"].sum().between(0.999999, 1.000001).all(), final_selected.groupby("method")["shadow_weight"].sum().to_dict())
expected_executable_sum = gate_exposure
audit("Final executable weights obey common gate", np.allclose(final_selected.groupby("method")["executable_weight"].sum(), expected_executable_sum, atol=1e-8), expected_executable_sum)
audit("No live-capital or quantum-advantage claim", not manifest["live_capital_authorized"] and not manifest["quantum_advantage_claimed"], "both false")
audit("All principal figures rendered", figure_created, figure_created)
audit("No NaN in principal performance metrics", not selected_periods[["cumulative_return", "annualized_return", "annualized_volatility", "sharpe_zero_rf", "maximum_drawdown"]].isna().any().any(), "checked")

audit_table = pd.DataFrame(audit_rows)
display(audit_table)
display(Markdown(f"**Passed:** {int(audit_table['passed'].sum())}/{len(audit_table)} checks."))


In [ ]:

assert audit_table["passed"].all(), audit_table.loc[~audit_table["passed"]].to_dict("records")
print("RESEARCH_AUDIT_FINAL_OK")


## 16. Đóng gói kết quả chạy

Cell cuối tạo một ZIP gồm manifest, toàn bộ CSV kết quả và hình ảnh. Mặc định notebook chỉ tạo file trong runtime; đổi `DOWNLOAD_RESULTS` thành `True` nếu muốn trình duyệt tự tải ZIP sau khi `Run all` hoàn tất.


In [ ]:
import shutil
from pathlib import Path

EXPORT_BASE = WORKDIR / "AUR_QAUR_Data_NCKH_Audit_5_9_RESULTS"
EXPORT_ZIP = Path(shutil.make_archive(str(EXPORT_BASE), "zip", root_dir=RESULTS))

print("Đã tạo gói kết quả:", EXPORT_ZIP)
print("Dung lượng:", f"{EXPORT_ZIP.stat().st_size / 1024**2:.2f} MiB")
print("Số tệp kết quả:", len([p for p in RESULTS.rglob("*") if p.is_file()]))

DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(EXPORT_ZIP))
else:
    print("Đặt DOWNLOAD_RESULTS = True và chạy lại cell này nếu muốn tải ZIP về máy.")
